In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from datetime import datetime

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  # preset only 2 to the embedding generation and leave 2 for other works
import torch

In [3]:
# Import everything from the core module
from moe_flashattn_3_core import (
    # Configurations
    BaseConfig,
    FlashAttentionConfig,
    MoEConfig,
    DownstreamConfig,
    
    # Models
    BaselineTransformer,
    FlashAttentionTransformer,
    FlashMoETransformer,
    
    # Data utilities
    ClinicalDataset,
    create_collate_fn,
    conv_cd,
    conv_age_gender,
    conv_lob,
    conv_target,
    
    # Embedding extraction
    EmbeddingExtractor,
    DownstreamEvaluator,
    
    # Model loading
    load_trained_model,
    get_experiment_configs,
    
    # GPU utilities
    cleanup_gpu_memory,
    cleanup_gpu_memory_hard,
    
    # Downstream evaluation
    run_downstream_evaluation_from_saved_model,
    run_multi_lob_downstream_evaluation,
    LOBData,
    
    # Logging
    MetricsLogger,
)
from torch.utils.data import DataLoader

#### Small test

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = "logs/exp_round5_3lobs_pretrain_multi_gpu_test_v2/exp6_auxiliary_free_v3/saved_models/exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp6_auxiliary_free_bs128_ep1_d256_20251231_152438_final.pt"
checkpoint_data = torch.load(checkpoint_path, map_location=device, weights_only=False)


Model type: FlashMoETransformer
Epoch: None
Global step: None


In [14]:
checkpoint_data.keys()

dict_keys(['model_state_dict', 'model_name', 'model_type', 'embedding_size', 'nlayers', 'checkpoint_dir', 'timestamp', 'config', 'moe_config'])

In [7]:
print(f"Model type: {checkpoint_data.get('model_type')}")
print(f"Epoch: {checkpoint_data.get('moe_config')}")
print(f"Global step: {checkpoint_data.get('config')}")

Model type: FlashMoETransformer
Epoch: {'d_model': 256, 'd_ff': 512, 'num_experts': 8, 'num_shared_experts': 1, 'top_k': 2, 'expert_dropout': 0.1, 'load_balance_strategy': 'deepseek', 'aux_loss_weight': 0.001, 'bias_lr': 0.003, 'bias_momentum': 0.6, 'z_loss_weight': 0.005, 'use_moe_from_layer': 2, 'use_swiglu_experts': True, 'router_warmup_steps': 0}
Global step: {'embedding_size': 256, 'nhid': 704, 'nhead': 8, 'nlayers': 6, 'dropout': 0.1}


In [38]:
# Load and inspect the checkpoint
model_path = "logs/exp_round5_3lobs_pretrain_multi_gpu_test_v2/exp6_auxiliary_free_v3/saved_models/exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp6_auxiliary_free_bs128_ep1_d256_20251231_152438_final.pt"

checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

print("="*80)
print("CHECKPOINT CONTENTS:")
print("="*80)

print("\n1. Top-level keys:")
for key in checkpoint.keys():
    print(f"  - {key}")

print("\n2. Config dict:")
config_dict = checkpoint.get('config', {})
for k, v in config_dict.items():
    print(f"  {k}: {v}")

print("\n3. MoE config dict:")
moe_config_dict = checkpoint.get('moe_config', None)
if moe_config_dict:
    for k, v in moe_config_dict.items():
        print(f"  {k}: {v}")
else:
    print("  ⚠️ moe_config is None!")

print("\n4. Model type:", checkpoint.get('model_type'))

# Calculate expected d_ff_adjusted for verification
if moe_config_dict and 'd_ff' in moe_config_dict:
    d_ff = moe_config_dict['d_ff']
    d_ff_adjusted = int((2 * d_ff) / 3)
    print(f"\n5. Expected d_ff_adjusted from checkpoint: {d_ff_adjusted}")

# Also check the actual weight shapes
print("\n6. Sample weight shapes from state_dict:")
state_dict = checkpoint['model_state_dict']
for key in list(state_dict.keys())[:10]:
    print(f"  {key}: {state_dict[key].shape}")

# Check one of the problematic layers
if 'temporal_layers.2.ffn.experts.0.ffn.w_gate.weight' in state_dict:
    shape = state_dict['temporal_layers.2.ffn.experts.0.ffn.w_gate.weight'].shape
    print(f"\n7. Expert FFN weight shape: {shape}")
    print(f"   This implies d_ff_adjusted = {shape[0]}")
    print(f"   This implies d_ff = {int(shape[0] * 3 / 2)}")

CHECKPOINT CONTENTS:

1. Top-level keys:
  - model_state_dict
  - model_name
  - model_type
  - embedding_size
  - nlayers
  - checkpoint_dir
  - timestamp
  - config
  - moe_config

2. Config dict:
  embedding_size: 256
  nhid: 704
  nhead: 8
  nlayers: 6
  dropout: 0.1

3. MoE config dict:
  d_model: 256
  d_ff: 512
  num_experts: 8
  num_shared_experts: 1
  top_k: 2
  expert_dropout: 0.1
  load_balance_strategy: deepseek
  aux_loss_weight: 0.001
  bias_lr: 0.003
  bias_momentum: 0.6
  z_loss_weight: 0.005
  use_moe_from_layer: 2
  use_swiglu_experts: True
  router_warmup_steps: 0

4. Model type: FlashMoETransformer

5. Expected d_ff_adjusted from checkpoint: 341

6. Sample weight shapes from state_dict:
  embedding_cd.weight: torch.Size([75516, 256])
  embedding_gender_cd.weight: torch.Size([4, 256])
  embedding_age_in_months.weight: torch.Size([1440, 256])
  embedding_lob.weight: torch.Size([4, 256])
  daily_pooling.query: torch.Size([1, 1, 256])
  daily_pooling.k_proj.weight: torc

### Intrinsic metrics eval

In [4]:
import json
import pandas as pd
pd.set_option('display.max_rows', None)
from pathlib import Path
from typing import Dict, List, Union, Optional

def extract_experiment_metrics(json_paths: Union[str, List[str]]) -> pd.DataFrame:
    """
    Extract all metrics from experiment result JSON files into a flat DataFrame.
    
    Handles:
    - Top-level summary metrics
    - full_evaluation.performance metrics
    - full_evaluation.efficiency metrics  
    - full_evaluation.resources metrics
    - all_epochs metrics (from final epoch)
    
    Args:
        json_paths: Single path or list of paths to result JSON files
        
    Returns:
        DataFrame with one row per experiment, all metrics as columns
    """
    if isinstance(json_paths, str):
        json_paths = [json_paths]
    
    all_records = []
    
    for path in json_paths:
        with open(path, 'r') as f:
            data = json.load(f)
        
        record = {}
        
        # ==================================================================
        # 1. TOP-LEVEL SUMMARY METRICS
        # ==================================================================
        top_level_keys = [
            'experiment', 'parameters', 'use_learned_pooling', 'use_bucketing',
            'train_loss_mean', 'train_loss_learned', 'train_loss_final',
            'val_loss_final', 'generalization_gap', 'training_time_sec',
            'final_train_recall@5', 'final_train_recall@10', 'final_train_recall@20',
            'final_val_recall@5', 'final_val_recall@10', 'final_val_recall@20',
            'final_val_micro_recall@10', 'final_val_ndcg@20', 'final_val_mrr',
            'final_val_positive_brier',
            'precision@10', 'recall@10', 'f1@10', 'micro_recall@10', 'ndcg@10',
            'balanced_top10_acc', 'tail_top10_acc', 'cost_usd', 'peak_memory_gb',
            'model_name'
        ]
        for key in top_level_keys:
            if key in data:
                record[key] = data[key]
        
        # ==================================================================
        # 2. full_evaluation.performance METRICS
        #    Source: comprehensive_evaluation() -> StreamingMetrics + detailed metrics
        # ==================================================================
        if 'full_evaluation' in data and 'performance' in data['full_evaluation']:
            perf = data['full_evaluation']['performance']
            for key, value in perf.items():
                record[f'perf_{key}'] = value
        
        # ==================================================================
        # 3. full_evaluation.efficiency METRICS
        #    Source: compute_training_time_metrics()
        # ==================================================================
        if 'full_evaluation' in data and 'efficiency' in data['full_evaluation']:
            eff = data['full_evaluation']['efficiency']
            for key, value in eff.items():
                record[f'eff_{key}'] = value
        
        # ==================================================================
        # 4. full_evaluation.resources METRICS
        #    Source: compute_memory_metrics() + compute_cost_metrics() + compute_flops_metrics()
        # ==================================================================
        if 'full_evaluation' in data and 'resources' in data['full_evaluation']:
            res = data['full_evaluation']['resources']
            for key, value in res.items():
                record[f'res_{key}'] = value
        
        # ==================================================================
        # 5. all_epochs METRICS (from final epoch)
        #    Source: _build_epoch_metrics() - includes training trajectory,
        #    eval-in-train, validation, MoE, and router metrics
        # ==================================================================
        if 'all_epochs' in data and len(data['all_epochs']) > 0:
            final_epoch = data['all_epochs'][-1]  # Get last epoch
            for key, value in final_epoch.items():
                record[f'epoch_{key}'] = value
        
        record['source_file'] = Path(path).name
        all_records.append(record)
    
    df = pd.DataFrame(all_records)
    
    # Reorder columns for readability
    priority_cols = ['experiment', 'source_file', 'parameters']
    other_cols = [c for c in df.columns if c not in priority_cols]
    df = df[priority_cols + sorted(other_cols)]
    
    return df

In [16]:
# Check the experiment round 5 intrinsic metrics
paths = [
    "logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_pure_legacy/saved_models/exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs128_ep1_d256_20251230_055716_results.json",
    "logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_opt_config/saved_models/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs64_ep1_d256_20260108_183616_results.json",
    "logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp2b_flash_learned_pool_v2/saved_models/exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20251230_114137_results.json",
    "logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp6_auxiliary_free_v3/saved_models/exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp6_auxiliary_free_bs128_ep1_d256_20251231_152438_results.json"
]
df_exp_round5_results = extract_experiment_metrics(paths)

In [8]:
df_exp_round5_results.T.to_excel("experiment_logs/exp_round5_3lobs_1-5M_1epoch_32batch_dim256_pretrain_multi_gpu_test_v2_intrinsic.xlsx")

### Model reconstruction

In [5]:
def load_model_from_checkpoint(
    model_path: str,
    device: torch.device,
    verbose: bool = True
) -> Tuple[torch.nn.Module, BaseConfig, Optional[MoEConfig]]:
    """
    Load a pretrained model from a checkpoint file.
    'model_state_dict', 'model_name', 'model_type', 'embedding_size', 'nlayers', 
    'checkpoint_dir', 'timestamp', 'config', 'moe_config'
    
    Args:
        model_path: Path to the .pt checkpoint file
        device: Torch device to load the model onto
        verbose: Whether to print loading details
        
    Returns:
        Tuple of (model, config, moe_config)
        - model: Loaded and initialized model in eval mode
        - config: Reconstructed FlashAttentionConfig or BaseConfig
        - moe_config: MoEConfig if model is MoE, else None
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"Loading model from: {model_path}")
    checkpoint_data = torch.load(model_path, map_location=device, weights_only=False)

    model_type = checkpoint_data.get('model_type', 'Unknown')
    config_dict = checkpoint_data.get('config', {})
    moe_config_dict = checkpoint_data.get('moe_config', None)
    
    state_dict = checkpoint_data['model_state_dict']
    
    if verbose:
        print(f"  Model type: {model_type}")
        print(f"  Embedding size: {config_dict.get('embedding_size', 256)}")
        print(f"  N layers: {config_dict.get('nlayers', 6)}")
        print(f"  Use learned attention pooling: {config_dict.get('use_learnt_att_pool', False)}")

    use_learnt_att_pool_inferred = 'daily_pooling.query' in state_dict    
    inferred_d_ff = None
    if 'FlashMoE' in model_type:
        # Try to infer d_ff from expert weight shapes
        # Look for: temporal_layers.{layer}.ffn.experts.0.ffn.w_gate.weight
        for key in state_dict.keys():
            if 'experts.0.ffn.w_gate.weight' in key:
                weight_shape = state_dict[key].shape
                d_ff_adjusted = weight_shape[0]  # Shape is [d_ff_adjusted, d_model]
                # Reverse the SwiGLU adjustment: d_ff = d_ff_adjusted * 3 / 2
                inferred_d_ff = (d_ff_adjusted * 3 + 1) // 2
                if verbose:
                    print(f"Inferred d_ff from expert weights: {inferred_d_ff} "
                          f"(d_ff_adjusted={d_ff_adjusted})")
                break
        
        # Alternative: use config.nhid which was correct
        if inferred_d_ff is None:
            inferred_d_ff = config_dict.get('nhid', 512)
            if verbose:
                print(f"Using nhid as d_ff fallback: {inferred_d_ff}")
        
        
    # Reconstruct config based on model type
    moe_config = None
    
    if 'FlashMoE' in model_type:
        # MoE model with Flash Attention
        config = FlashAttentionConfig(
            embedding_size=config_dict.get('embedding_size', 256),
            nhid=config_dict.get('nhid', 512),
            nhead=config_dict.get('nhead', 8),
            nlayers=config_dict.get('nlayers', 6),
            dropout=config_dict.get('dropout', 0.1),
            use_learnt_att_pool=use_learnt_att_pool_inferred,
            use_swiglu=config_dict.get('use_swiglu', True),
            use_rope=config_dict.get('use_rope', True),
            use_flash=config_dict.get('use_flash', True),
        )
        
        if moe_config_dict:
            d_ff_to_use = inferred_d_ff or config_dict.get('nhid', 512)
            
            if verbose and moe_config_dict.get('d_ff') != d_ff_to_use:
                print(f"⚠️ Correcting d_ff: checkpoint has {moe_config_dict.get('d_ff')}, "
                      f"using {d_ff_to_use}")
            moe_config = MoEConfig(
                d_model=moe_config_dict.get('d_model', config.embedding_size),
                d_ff=d_ff_to_use,
                num_experts=moe_config_dict.get('num_experts', 8),
                num_shared_experts=moe_config_dict.get('num_shared_experts', 1),
                top_k=moe_config_dict.get('top_k', 2),
                expert_dropout=moe_config_dict.get('expert_dropout', 0.1),
                load_balance_strategy=moe_config_dict.get('load_balance_strategy', 'deepseek'),
                aux_loss_weight=moe_config_dict.get('aux_loss_weight', 0.001),
                use_moe_from_layer=moe_config_dict.get('use_moe_from_layer', 2),
                use_swiglu_experts=moe_config_dict.get('use_swiglu_experts', True),
                router_warmup_steps=moe_config_dict.get('router_warmup_steps', 0),
                z_loss_weight=moe_config_dict.get('z_loss_weight', 0.005),
                bias_lr=moe_config_dict.get('bias_lr', 1e-3),
                bias_momentum=moe_config_dict.get('bias_momentum', 0.6),
                
            )
            if verbose:
                print(f"  MoE config: {moe_config_dict.get('num_experts')} experts, "
                      f"top-{moe_config_dict.get('top_k')}, from layer {moe_config_dict.get('use_moe_from_layer')}")
        else:
            moe_config = MoEConfig(d_model=config.embedding_size, d_ff=config.nhid)
        
        model = FlashMoETransformer(config, moe_config)
        use_mixed_precision = True
        
    elif 'FlashAttention' in model_type:
        # Flash Attention model (no MoE)
        config = FlashAttentionConfig(
            embedding_size=config_dict.get('embedding_size', 256),
            nhid=config_dict.get('nhid', 512),
            nhead=config_dict.get('nhead', 8),
            nlayers=config_dict.get('nlayers', 6),
            dropout=config_dict.get('dropout', 0.1),
            use_learnt_att_pool=config_dict.get('use_learnt_att_pool', True),
            use_swiglu=config_dict.get('use_swiglu', True),
            use_rope=config_dict.get('use_rope', True),
            use_flash=config_dict.get('use_flash', True),
        )
        model = FlashAttentionTransformer(config)
        use_mixed_precision = True
        
    else:
        # Baseline Transformer
        config = BaseConfig(
            embedding_size=config_dict.get('embedding_size', 256),
            nhid=config_dict.get('nhid', 512),
            nlayers=config_dict.get('nlayers', 6),
            dropout=config_dict.get('dropout', 0.1),
        )
        model = BaselineTransformer(config)
        use_mixed_precision = False
        
    # Load state dict
    model.load_state_dict(checkpoint_data['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    total_params = sum(p.numel() for p in model.parameters())
    if verbose:
        total_params = sum(p.numel() for p in model.parameters())
        print(f"✅ Model loaded successfully!")
        print(f"   Total parameters: {total_params:,}")
        print(f"   Mixed precision: {use_mixed_precision}")
        print(f"   Device: {device}")
        print(f"{'='*80}\n")
    
    
    return model, config, moe_config, use_mixed_precision, model_type

In [6]:
from torch.utils.data import Dataset
class LazyClinicalDataset(Dataset):
    """
    Memory-efficient dataset that parses data on-the-fly.
    
    Memory usage: O(1) instead of O(N)
    Trade-off: Slightly slower due to per-batch parsing, but negligible
    compared to GPU inference time.
    
    Perfect for inference on large datasets.
    """
    
    def __init__(self, df: pd.DataFrame, config: BaseConfig):
        self.config = config
        # Store only the DataFrame reference - NO tensor allocation!
        self.df = df.reset_index(drop=True)
        
        # Pre-extract columns as lists for faster access
        self.age_strs = self.df['age_in_months'].tolist()
        self.gender_strs = self.df['gender_cd'].tolist()
        self.cd_strs = self.df['cd'].tolist()
        self.lob_strs = self.df['lob'].tolist()
        self.dt_cnt = self.df['dt_cnt'].tolist()
        
        # Only needed for training - can skip for inference
        self.target_strs = self.df['target'].tolist() if 'target' in self.df.columns else None
        
        print(f"LazyClinicalDataset initialized with {len(self.df):,} samples (lazy loading)")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Parse data on-demand (not pre-loaded!)
        age_list = conv_age_gender(self.age_strs[idx], self.config.len_dy)
        gender_list = conv_age_gender(self.gender_strs[idx], self.config.len_dy, max_val=3)
        cd_list = conv_cd(self.cd_strs[idx], self.config.len_dy, self.config.len_cd)
        lob_list = conv_lob(self.lob_strs[idx], self.config.len_dy)
        
        # Convert to tensors
        age = torch.tensor(age_list, dtype=torch.long)
        gender = torch.tensor(gender_list, dtype=torch.long)
        codes = torch.tensor(cd_list, dtype=torch.long)
        lob = torch.tensor(lob_list, dtype=torch.long)
        
        # Target (for training) - lazy parse
        if self.target_strs is not None:
            target_list = conv_target(self.target_strs[idx], self.config.len_dy, self.config.target_cd_cnt)
        else:
            target_list = [[0] for _ in range(self.config.len_dy)]
        
        return {
            'age': age,
            'gender': gender,
            'lob': lob,
            'codes': codes,
            'dt_cnt': self.dt_cnt[idx],
            'target': target_list
        }

### Generate embeddings

In [7]:
import time
import copy
from typing import Tuple, List, Optional
from tqdm import tqdm
from torch.utils.data import DataLoader
from concurrent.futures import ThreadPoolExecutor
import threading

def generate_embeddings(
    model: torch.nn.Module,
    config: 'BaseConfig',
    data: pd.DataFrame,
    device: torch.device,
    id_column: str = 'individual_id',       # Parameterized: ID column name
    lob_value: Optional[str] = None,         # Parameterized: Auto-add LOB if specified
    desc_prefix: str = '',                   # Parameterized: Progress bar prefix
    batch_size: int = 64,
    num_workers: int = 4,
    use_mixed_precision: bool = True,
    verbose: bool = True,
    multi_gpu: bool = False,
    moe_config: Optional['MoEConfig'] = None,
) -> Tuple[np.ndarray, List[str], List[str]]:
    """
    Generate embeddings for all members in the dataset (unified for all LOBs).
    
    This is the consolidated embedding generation function that works for:
    - Commercial: id_column='individual_id', lob_value=None (already in data)
    - Medicaid: id_column='asdb_member_key', lob_value='Medicaid'
    - Medicare: id_column='individual_id', lob_value='Medicare' (if needed)
    
    The embedding is extracted from the FINAL TEMPORAL REPRESENTATION:
    - BaselineTransformer: output of transformer_encoder_dy
    - FlashAttentionTransformer: input to model.norm (after all temporal layers)
    - FlashMoETransformer: input to model.norm (after all temporal + MoE layers)
    
    Patient embedding = embedding at the LAST VALID DAY (dt_cnt - 1)
    
    Optimizations:
    1. Pre-allocated pinned memory output (no vstack)
    2. Non-blocking async GPU→CPU transfers
    3. torch.inference_mode (faster than no_grad)
    4. Optional multi-GPU with true parallelism
    5. Progress bar with ETA
    
    Args:
        model: Loaded model in eval mode
        config: Model configuration
        data: DataFrame with required columns (age_in_months, gender_cd, cd, lob, dt_cnt, etc.)
        device: Primary device
        id_column: Column name for member IDs (default: 'individual_id')
        lob_value: If specified, add this as 'lob' column if missing (e.g., 'Medicaid')
        desc_prefix: Prefix for progress bar description (e.g., 'Medicaid')
        batch_size: Batch size per GPU
        num_workers: DataLoader workers
        use_mixed_precision: Use FP16 for Flash models
        verbose: Print progress
        multi_gpu: Enable multi-GPU processing
        moe_config: MoE config (required for multi-GPU with MoE models)
        
    Returns:
        embeddings: np.ndarray [num_members, embedding_size]
        member_ids: List of member IDs (from id_column)
        index_dts: List of index dates
    """
    start_time = time.time()
    n_samples = len(data)
    embedding_dim = config.embedding_size
    
    # Detect model type
    has_moe = (hasattr(model, 'forward') and 
               'return_moe_losses' in model.forward.__code__.co_varnames)
    
    n_gpus = torch.cuda.device_count() if multi_gpu else 1
    
    # Build description
    desc = f"{desc_prefix} " if desc_prefix else ""
    desc += "Embedding Generation"
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"{desc.upper()}")
        print(f"{'='*70}")
        print(f"Samples: {n_samples:,} | Batch: {batch_size} | GPUs: {n_gpus}")
        print(f"Workers: {num_workers} | Mixed precision: {use_mixed_precision}")
        print(f"ID column: {id_column}")
    
    # Handle LOB column (for Medicaid/Medicare where it may not exist)
    if lob_value and 'lob' not in data.columns:
        data = data.copy()
        data['lob'] = lob_value
        if verbose:
            print(f"  Added 'lob'='{lob_value}' column")
    
    # Pre-allocate pinned memory output
    embeddings_output = torch.empty(
        (n_samples, embedding_dim),
        dtype=torch.float32,
        pin_memory=True
    )
    
    # Extract IDs using the specified column
    if id_column in data.columns:
        member_ids = data[id_column].astype(str).tolist()
    else:
        # Fallback to individual_id if specified column not present
        member_ids = data['individual_id'].astype(str).tolist()
        if verbose:
            print(f"  Warning: '{id_column}' not found, using 'individual_id'")
    
    index_dts = data['index_dt'].astype(str).tolist()
    
    # Build progress description
    pbar_desc = f"Generating {desc_prefix} embeddings" if desc_prefix else "Generating embeddings"
    
    if n_gpus > 1 and multi_gpu:
        return _generate_embeddings_multi_gpu(
            model=model,
            config=config,
            data=data,
            embeddings_output=embeddings_output,
            member_ids=member_ids,
            index_dts=index_dts,
            n_gpus=n_gpus,
            batch_size=batch_size,
            num_workers=num_workers,
            use_mixed_precision=use_mixed_precision,
            has_moe=has_moe,
            moe_config=moe_config,
            verbose=verbose,
            start_time=start_time,
            pbar_desc=pbar_desc,
        )
    else:
        return _generate_embeddings_single_gpu(
            model=model,
            config=config,
            data=data,
            device=device,
            embeddings_output=embeddings_output,
            member_ids=member_ids,
            index_dts=index_dts,
            batch_size=batch_size,
            num_workers=num_workers,
            use_mixed_precision=use_mixed_precision,
            has_moe=has_moe,
            verbose=verbose,
            start_time=start_time,
            pbar_desc=pbar_desc,
        )


def _generate_embeddings_single_gpu(
    model, config, data, device, embeddings_output,
    member_ids, index_dts, batch_size, num_workers,
    use_mixed_precision, has_moe, verbose, start_time,
    pbar_desc: str = "Generating embeddings"
) -> Tuple[np.ndarray, List[str], List[str]]:
    """Single GPU optimized path (unified for all LOBs)."""
    
    n_samples = len(data)
    model.eval()
    
    dataset = LazyClinicalDataset(data, config)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=create_collate_fn(config),
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=num_workers > 0,
    )
    
    current_idx = 0
    pbar = tqdm(dataloader, desc=pbar_desc, disable=not verbose)
    
    with torch.inference_mode():
        with EmbeddingExtractor(model) as extractor:
            for batch in pbar:
                batch_size_actual = batch['age'].shape[0]
                batch_start = current_idx
                batch_end = batch_start + batch_size_actual
                
                x = torch.cat([
                    batch['age'].unsqueeze(-1),
                    batch['gender'].unsqueeze(-1),
                    batch['lob'].unsqueeze(-1),
                    batch['codes']
                ], dim=-1).to(device, non_blocking=True)
                
                dt_cnt = batch['dt_cnt']
                
                # Forward pass
                if use_mixed_precision:
                    with torch.cuda.amp.autocast(dtype=torch.float16):
                        if has_moe:
                            _ = model(x, return_moe_losses=False)
                        else:
                            _ = model(x)
                else:
                    if has_moe:
                        _ = model(x, return_moe_losses=False)
                    else:
                        _ = model(x)
                
                # Extract embeddings
                dt_cnt_list = dt_cnt.tolist() if isinstance(dt_cnt, torch.Tensor) else dt_cnt
                patient_embs = extractor.get_patient_embedding(dt_cnt_list)
                
                # Async copy to pre-allocated pinned memory
                embeddings_output[batch_start:batch_end].copy_(
                    patient_embs.float(),
                    non_blocking=True
                )
                
                current_idx = batch_end
                
                # Progress metrics
                elapsed = time.time() - start_time
                speed = batch_end / elapsed
                eta = (n_samples - batch_end) / speed if speed > 0 else 0
                pbar.set_postfix({
                    'speed': f'{speed:.0f}/s',
                    'ETA': f'{eta:.0f}s'
                })
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    embeddings = embeddings_output.numpy()
    
    elapsed = time.time() - start_time
    if verbose:
        print(f"\n✅ Complete! Time: {elapsed:.1f}s | Speed: {n_samples/elapsed:,.0f} samples/s")
        print(f"   Output: {embeddings.shape}")
    
    return embeddings, member_ids, index_dts


def _generate_embeddings_multi_gpu(
    model, config, data, embeddings_output, member_ids, index_dts,
    n_gpus, batch_size, num_workers, use_mixed_precision, has_moe,
    moe_config, verbose, start_time,
    pbar_desc: str = "Multi-GPU"
) -> Tuple[np.ndarray, List[str], List[str]]:
    """Multi-GPU path with true parallelism (unified for all LOBs)."""
    
    n_samples = len(data)
    
    if verbose:
        print(f"Multi-GPU mode: {n_gpus} GPUs")
    
    # Clone model to each GPU
    models = []
    for gpu_id in range(n_gpus):
        if verbose:
            print(f"  Cloning model to GPU {gpu_id}...")
        
        with torch.cuda.device(gpu_id):
            model_copy = copy.deepcopy(model)
            model_copy = model_copy.to(f'cuda:{gpu_id}')
            model_copy.eval()
            models.append(model_copy)
    
    # Split data into chunks
    chunk_size = (n_samples + n_gpus - 1) // n_gpus
    data_chunks = []
    start_indices = []
    
    for i in range(n_gpus):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, n_samples)
        data_chunks.append(data.iloc[start_idx:end_idx].reset_index(drop=True))
        start_indices.append(start_idx)
        
        if verbose:
            print(f"  GPU {i}: samples {start_idx:,} to {end_idx:,} ({end_idx - start_idx:,} samples)")
    
    progress_lock = threading.Lock()
    total_processed = [0]
    errors = []
    
    def process_chunk(gpu_id: int, data_chunk: pd.DataFrame, start_idx: int):
        """Process a data chunk on a specific GPU."""
        if len(data_chunk) == 0:
            return
        
        try:
            gpu_device = torch.device(f'cuda:{gpu_id}')
            gpu_model = models[gpu_id]
            
            dataset = LazyClinicalDataset(data_chunk, config)
            dataloader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=create_collate_fn(config),
                num_workers=max(1, num_workers // n_gpus),
                pin_memory=True,
            )
            
            local_idx = start_idx
            
            with torch.inference_mode():
                with EmbeddingExtractor(gpu_model) as extractor:
                    for batch in dataloader:
                        batch_size_actual = batch['age'].shape[0]
                        
                        x = torch.cat([
                            batch['age'].unsqueeze(-1),
                            batch['gender'].unsqueeze(-1),
                            batch['lob'].unsqueeze(-1),
                            batch['codes']
                        ], dim=-1).to(gpu_device, non_blocking=True)
                        
                        dt_cnt = batch['dt_cnt']
                        
                        if use_mixed_precision:
                            with torch.cuda.amp.autocast(dtype=torch.float16):
                                if has_moe:
                                    _ = gpu_model(x, return_moe_losses=False)
                                else:
                                    _ = gpu_model(x)
                        else:
                            if has_moe:
                                _ = gpu_model(x, return_moe_losses=False)
                            else:
                                _ = gpu_model(x)
                        
                        dt_cnt_list = dt_cnt.tolist() if isinstance(dt_cnt, torch.Tensor) else dt_cnt
                        patient_embs = extractor.get_patient_embedding(dt_cnt_list)
                        
                        embeddings_output[local_idx:local_idx + batch_size_actual].copy_(
                            patient_embs.float(),
                            non_blocking=True
                        )
                        
                        local_idx += batch_size_actual
                        
                        with progress_lock:
                            total_processed[0] += batch_size_actual
            
            torch.cuda.synchronize(gpu_device)
            
        except Exception as e:
            errors.append((gpu_id, str(e)))
    
    # Launch parallel processing
    if verbose:
        pbar = tqdm(total=n_samples, desc=f"{pbar_desc} ({n_gpus} GPUs)")
    
    with ThreadPoolExecutor(max_workers=n_gpus) as executor:
        futures = [
            executor.submit(process_chunk, gpu_id, data_chunks[gpu_id], start_indices[gpu_id])
            for gpu_id in range(n_gpus)
        ]
        
        last_count = 0
        while not all(f.done() for f in futures):
            with progress_lock:
                current = total_processed[0]
            if verbose:
                pbar.update(current - last_count)
            last_count = current
            time.sleep(0.1)
        
        if verbose:
            pbar.update(n_samples - last_count)
            pbar.close()
        
        for f in futures:
            f.result()
    
    if errors:
        raise RuntimeError(f"GPU errors: {errors}")
    
    # Cleanup
    for m in models:
        del m
    torch.cuda.empty_cache()
    
    embeddings = embeddings_output.numpy()
    
    elapsed = time.time() - start_time
    if verbose:
        print(f"\n✅ Complete! Time: {elapsed:.1f}s | Speed: {n_samples/elapsed:,.0f} samples/s")
        print(f"   Effective: {n_samples/elapsed * n_gpus:,.0f} samples/s (across {n_gpus} GPUs)")
        print(f"   Output: {embeddings.shape}")
    
    return embeddings, member_ids, index_dts

#### Save embeddings

In [8]:
def save_embeddings(
    embeddings: np.ndarray,
    individual_ids: List[str],
    index_dts: List[str],
    output_path: str,
    model_name: str = "",
    additional_metadata: Dict = None
) -> str:
    """
    Save embeddings to disk in a structured format.
    
    Saves both:
    1. NPZ file with embeddings and metadata
    2. CSV file with IDs and index dates for easy lookup
    
    Args:
        embeddings: [num_members, embedding_dim] array
        individual_ids: List of member IDs
        index_dts: List of index dates
        output_path: Directory to save files
        model_name: Name of the model for file naming
        additional_metadata: Optional dict of additional info to save
        
    Returns:
        Path to saved NPZ file
    """
    os.makedirs(output_path, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Generate filename
    if model_name:
        filename_base = f"embeddings_{model_name}_{timestamp}"
    else:
        filename_base = f"embeddings_{timestamp}"
    
    # Save NPZ with embeddings and metadata
    npz_path = os.path.join(output_path, f"{filename_base}.npz")
    np.savez_compressed(
        npz_path,
        embeddings=embeddings,
        individual_ids=np.array(individual_ids, dtype=object),
        index_dts=np.array(index_dts, dtype=object),
        embedding_dim=embeddings.shape[1],
        num_members=len(individual_ids),
        **(additional_metadata or {})
    )
    print(f"Embeddings saved to: {npz_path}")
    
    # Save CSV for easy lookup
    csv_path = os.path.join(output_path, f"{filename_base}_ids.csv")
    pd.DataFrame({
        'individual_id': individual_ids,
        'index_dt': index_dts,
        'embedding_idx': range(len(individual_ids))
    }).to_csv(csv_path, index=False)
    print(f"ID mapping saved to: {csv_path}")
    
    return npz_path

In [9]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import os

def save_embeddings_to_bigquery(
    embeddings: np.ndarray,
    individual_ids: list,
    index_dts: list,
    project_id: str,
    dataset_id: str,
    table_name: str,
    exp_name: str = "",
    model_type: str = "",
    if_exists: str = "replace",  # 'replace', 'append', 'fail'
    max_bytes_per_chunk: int = 500_000_000,  # ~500 MB target per chunk
) -> str:
    """
    Save embeddings to BigQuery.
    
    Automatically chunks uploads when the estimated payload exceeds
    max_bytes_per_chunk, avoiding BigQuery memory-limit errors on
    wide/large embedding tables.
    
    Args:
        embeddings: numpy array [num_members, embedding_dim]
        individual_ids: list of member IDs
        index_dts: list of index dates
        project_id: GCP project ID
        dataset_id: BigQuery dataset ID
        table_name: Table name to create
        exp_name: Experiment name for metadata
        model_type: Model type for metadata
        if_exists: What to do if table exists ('replace', 'append', 'fail')
        max_bytes_per_chunk: Approximate max bytes per upload chunk.
            Defaults to 500 MB. Lower if you still hit memory limits.
        
    Returns:
        Full table path
    """
    n_total = len(individual_ids)
    embedding_dim = embeddings.shape[1]
    full_table_id = f"{project_id}.{dataset_id}.{table_name}"

    # Estimate row size: embedding floats + ~200 bytes overhead for ID/date/metadata columns
    bytes_per_row = embedding_dim * 4 + 200
    estimated_total_bytes = bytes_per_row * n_total
    chunk_size = max(1, max_bytes_per_chunk // bytes_per_row)
    n_chunks = (n_total + chunk_size - 1) // chunk_size

    print(f"Writing {n_total:,} rows to BigQuery: {full_table_id}")
    print(f"  Columns: {embedding_dim + 4} (embedding_dim={embedding_dim})")
    print(f"  Estimated payload: {estimated_total_bytes / 1e9:.2f} GB")
    if n_chunks > 1:
        print(f"  Chunking into {n_chunks} uploads of ~{chunk_size:,} rows each")

    client = bigquery.Client()

    # Map if_exists to write disposition for the *first* chunk
    first_disposition = {
        "replace": bigquery.WriteDisposition.WRITE_TRUNCATE,
        "append":  bigquery.WriteDisposition.WRITE_APPEND,
        "fail":    bigquery.WriteDisposition.WRITE_EMPTY,
    }[if_exists]

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, n_total)

        df_chunk = pd.DataFrame({
            'individual_id': individual_ids[start:end],
            'index_dt': index_dts[start:end],
        })
        for i in range(embedding_dim):
            df_chunk[f'embedding_{i}'] = embeddings[start:end, i].astype(np.float32)
        df_chunk['exp_name'] = exp_name
        df_chunk['model_type'] = model_type

        write_disp = first_disposition if chunk_idx == 0 else bigquery.WriteDisposition.WRITE_APPEND

        job_config = bigquery.LoadJobConfig(write_disposition=write_disp)
        job = client.load_table_from_dataframe(df_chunk, full_table_id, job_config=job_config)
        job.result()

        print(f"  ✓ Chunk {chunk_idx + 1}/{n_chunks}: rows [{start:,} – {end:,}) uploaded")
        del df_chunk

    table = client.get_table(full_table_id)
    print(f"✅ Loaded {table.num_rows:,} rows to {full_table_id}")

    return full_table_id

##### Test

In [40]:
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"
TABLE_NAME = "a964286_TEST_embedding_function_validation"
table_ref = client.dataset(DATASET_ID).table(TABLE_NAME)

In [47]:
# Create fake data
num_samples = 100
embedding_dim = 256

# Generate fake embeddings (random floats)
fake_embeddings = np.random.randn(num_samples, embedding_dim).astype(np.float32)

# Generate fake individual IDs
fake_individual_ids = [f"FAKE_ID_{i:06d}" for i in range(num_samples)]

# Generate fake index dates
fake_index_dts = pd.date_range(
    start='2023-01-01', 
    periods=num_samples, 
    freq='D'
).strftime('%Y-%m-%d').tolist()

# Test configuration
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"
TABLE_NAME = "a964286_TEST_embedding_function_validation"  # Use a clearly named test table
result_table = save_embeddings_to_bigquery(
    embeddings=fake_embeddings,
    individual_ids=fake_individual_ids,
    index_dts=fake_index_dts,
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_name=TABLE_NAME,
    exp_name="test_experiment",
    model_type="test_model",
    if_exists="replace"
)

Writing 100 rows to BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_TEST_embedding_function_validation
  Columns: 260 (embedding_dim=256)
✅ Loaded 100 rows to edp-prod-storage.edp_ent_sdoheir_cns.a964286_TEST_embedding_function_validation


### Feature importance

In [4]:
# =============================================================================
# SHARED FEATURE IMPORTANCE MODULE (SHAP — Model-Agnostic)
# =============================================================================
# Used by both Commercial and Medicaid sections.
# Goal: quantify what proportion of top-N important features are embeddings.

import shap

def compute_shap_feature_importance(
    fitted_model,
    X_eval: pd.DataFrame,
    feature_cols: List[str],
    embedding_features: List[str],
    top_k_list: List[int] = [10, 20, 50],
    max_samples: int = 2000,
    random_state: int = 42,
    model_name: str = "",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute SHAP feature importance and analyze embedding feature proportions.

    Works with any model that has a predict_proba method (LogisticRegression,
    CatBoost, XGBoost, LightGBM, etc.).

    Args:
        fitted_model: A TRAINED model with predict_proba().
        X_eval: Evaluation DataFrame (use val or test split, NOT train).
        feature_cols: Ordered list of feature column names matching X_eval columns.
        embedding_features: List of embedding column names (subset of feature_cols).
        top_k_list: List of top-K cutoffs for proportion analysis (default [10, 20, 50]).
        max_samples: Cap on background/eval samples for SHAP speed (default 2000).
        random_state: Seed for sampling reproducibility.
        model_name: Label for output (e.g. "CatBoost_hybrid").
        verbose: Print progress.

    Returns:
        shap_summary_df: DataFrame with columns [feature, mean_abs_shap, rank, is_embedding]
                         sorted by mean_abs_shap descending.
        proportion_df:   DataFrame with columns [model_name, top_k, n_embedding_in_top_k,
                         proportion_embedding, n_tabular_in_top_k, proportion_tabular]
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"SHAP FEATURE IMPORTANCE: {model_name or type(fitted_model).__name__}")
        print(f"{'='*70}")

    X_sample = X_eval
    if len(X_eval) > max_samples:
        X_sample = X_eval.sample(n=max_samples, random_state=random_state)
        if verbose:
            print(f"  Sampled {max_samples} rows from {len(X_eval)} for SHAP computation")

    model_type = type(fitted_model).__name__

    if model_type in ('CatBoostClassifier',):
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_sample)
    elif model_type in ('XGBClassifier', 'LGBMClassifier'):
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_sample)
    elif model_type == 'LogisticRegression':
        background = shap.sample(X_sample, min(100, len(X_sample)))
        explainer = shap.LinearExplainer(fitted_model, background)
        shap_values = explainer.shap_values(X_sample)
    else:
        background = shap.sample(X_sample, min(100, len(X_sample)))
        explainer = shap.KernelExplainer(
            fitted_model.predict_proba, background
        )
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    embedding_set = set(embedding_features)
    shap_summary_df = pd.DataFrame({
        'feature': feature_cols,
        'mean_abs_shap': mean_abs_shap,
        'is_embedding': [f in embedding_set for f in feature_cols],
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    shap_summary_df['rank'] = range(1, len(shap_summary_df) + 1)

    proportion_rows = []
    for k in top_k_list:
        k_actual = min(k, len(shap_summary_df))
        top_k_df = shap_summary_df.head(k_actual)
        n_emb = int(top_k_df['is_embedding'].sum())
        n_tab = k_actual - n_emb
        proportion_rows.append({
            'model_name': model_name or model_type,
            'top_k': k,
            'n_embedding_in_top_k': n_emb,
            'proportion_embedding': round(n_emb / k_actual, 4),
            'n_tabular_in_top_k': n_tab,
            'proportion_tabular': round(n_tab / k_actual, 4),
        })

    proportion_df = pd.DataFrame(proportion_rows)

    if verbose:
        print(f"\n  Top 20 features by mean |SHAP|:")
        print(shap_summary_df[['rank', 'feature', 'mean_abs_shap', 'is_embedding']].head(20).to_string(index=False))
        print(f"\n  Embedding Proportion Analysis:")
        print(proportion_df.to_string(index=False))

    return shap_summary_df, proportion_df


def run_shap_for_all_feature_sets(
    fitted_models: Dict[str, Any],
    prepared_data_dict: Dict[str, Any],
    top_k_list: List[int] = [10, 20, 50],
    max_samples: int = 2000,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run SHAP analysis across multiple (model, feature_set) combos.

    Args:
        fitted_models: Dict mapping label -> fitted model (must already be trained).
        prepared_data_dict: Dict mapping same labels -> prepared data objects
                            (PreparedData or MedicaidPreparedData).
        top_k_list: Top-K cutoffs.
        max_samples: SHAP sample cap.
        verbose: Print progress.

    Returns:
        all_shap_df: Concatenated SHAP summaries with 'experiment' column.
        all_proportion_df: Concatenated proportion summaries.
    """
    all_shap = []
    all_proportion = []

    for label, model in fitted_models.items():
        pd_obj = prepared_data_dict[label]

        if hasattr(pd_obj, 'X_splits'):
            X_eval = pd_obj.X_splits.get('test', pd_obj.X_splits.get('val'))
        elif hasattr(pd_obj, 'X_test'):
            X_eval = pd_obj.X_test
        else:
            raise ValueError(f"Cannot find evaluation data in {type(pd_obj)}")

        feature_cols = pd_obj.feature_cols
        embedding_features = pd_obj.embedding_features

        shap_df, prop_df = compute_shap_feature_importance(
            fitted_model=model,
            X_eval=X_eval,
            feature_cols=feature_cols,
            embedding_features=embedding_features,
            top_k_list=top_k_list,
            max_samples=max_samples,
            model_name=label,
            verbose=verbose,
        )
        shap_df['experiment'] = label
        all_shap.append(shap_df)
        all_proportion.append(prop_df)

    return pd.concat(all_shap, ignore_index=True), pd.concat(all_proportion, ignore_index=True)

### Commercial IP downstream

In [5]:
import google.auth
from google.auth import impersonated_credentials
from google.cloud import bigquery
client = bigquery.Client()
credentials, project= google.auth.default()
print('credentials:', credentials, ', project:', project)
import pandas as pd
from tqdm.notebook import tqdm
client = bigquery.Client()

credentials: <google.oauth2.credentials.Credentials object at 0x7f13553fc2e0> , project: edp-prod-css-sdoh


In [23]:
# import members not in the trainingset of the transformer
commercial_sql_code = """
select * from edp-prod-storage.edp_ent_sdoheir_cns.a964286_commercial_heldout_transformer_input_4_te_experiment_round_5
"""
df_cm = client.query(commercial_sql_code).to_dataframe()

In [24]:
# sample before and after 2023-10-16 (post part will be used for oot validation)
# 0.3 samples for efficent evaluations of embeddings
df_cm['index_dt'] = pd.to_datetime(df_cm['index_dt'])
df_cm_b4_oct = df_cm[df_cm['index_dt'] <= pd.to_datetime("2023-10-16")]
df_cm_after_oct = df_cm[df_cm['index_dt'] > pd.to_datetime("2023-10-16")]
df_cm_b4_oct_sample = df_cm_b4_oct.sample(frac=0.3, random_state=42)
df_cm_after_oct_sample = df_cm_after_oct.sample(frac=0.3, random_state=42)
df_cm_sample = pd.concat([df_cm_b4_oct_sample,
                         df_cm_after_oct])

#### Embedding generation

In [37]:
MODEL_PATHS = {
#     # Experiment 1: Dense Baseline (no Flash Attention, no MoE)
#     'exp1_dense_baseline_pure_legacy': 
#         'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_pure_legacy/saved_models/'
#         'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs128_ep1_d256_20251230_055716_final.pt',
    
    # Experiment 1b: Dense Baseline (same opt config as 2b and 6)
    # 'exp1_dense_baseline_opt_config': 
    #     'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_opt_config/saved_models/'
    #     'exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs64_ep1_d256_20260108_183616_final.pt',

    # Experiment 2b: Flash Attention + Learned Pooling (no MoE)
    # 'exp2b_flash_learned_pool_v2':
    #     'logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/'
    #     'exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt' 
    
    # Experiment 2b: plus 2 stage modeling + cooccurrence embedding
    'exp2b_flash_learned_pool_v2':
        'logs/exp_round9_3lobs_1-5M_decoupled_training_embedding_v4_256dim/exp2b_flash_learned_pool_v2/saved_models/'
        'exp_round9_3lobs_1-5M_decoupled_training_embedding_v4_256dim_exp2b_flash_learned_pool_bs128_ep1_d256_20260310_123547_final.pt' 
    
    # # Experiment 6: Flash + MoE with DeepSeek auxiliary-free balancing
    # 'exp6_auxiliary_free_v3': 
    #     'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/'
    #     'exp6_auxiliary_free_v3/saved_models/'
    #     'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp6_auxiliary_free_bs128_ep1_d256_20251231_152438_final.pt',

    
    
    # Experiment 2b: Flash Attention + Learned Pooling (no MoE) + focalloss_dense_sampler
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/'
    # 'exp2b_flash_learned_pool_v5_asym_focalloss_dense_sampler/saved_models/'
    # 'exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260226_014755_final.pt'

    # # Experiment 2: exp round 6 with 256 dimensions and 6.8M member; regular BCE with weights 200
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim/'
    # 'exp2b_flash_learned_pool/saved_models/'
    # 'exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final.pt'
    
    # Experiment 2: exp round 7 with 512 dimensions; regular BCE with weights 200
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim/'
    # 'exp2b_flash_learned_poolexp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final/saved_models/'
    # 'exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final.pt'
    
    # Experiment 2: exp round 6 with 256 dimensions; regular BCE with weights 200
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2/'
    # 'exp2b_flash_learned_pool_6-8M/saved_models/'
    # 'exp_round6_3lobs_6-8M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260304_221738_final.pt'
    # Round 5; 
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2/'
    # 'exp2b_flash_learned_pool/saved_models/'
    # 'exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260110_112709_final.pt'
    
    # Experiment 2b: plus 2 stage modeling on full book of business
    'exp2b_flash_learned_pool_v2':
        'logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/'
        'exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt' 
}

In [39]:
results = {}
batch_size = 64
# output_dir = "embedding_output/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2"
# output_dir = "embedding_output/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2"
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"
LOB = 'commercial'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for exp_name, model_path in tqdm(MODEL_PATHS.items()):
    cleanup_gpu_memory(verbose=False)
    model, config, moe_config, use_mixed_precision, model_type = load_model_from_checkpoint(
        model_path=MODEL_PATHS[exp_name],
        device=device,
        verbose=True
    )
    inference_start_time = time.time()
    embeddings, individual_ids, index_dts = generate_embeddings(
        model=model,
        config=config,
        data=df_cm_sample,
        device=device,
        id_column='individual_id',  # Commercial uses individual_id
        lob_value=None,              # Commercial data already has lob column
        desc_prefix='Commercial',
        batch_size=batch_size,
        use_mixed_precision=use_mixed_precision,
        verbose=True,
        multi_gpu=True,           
        moe_config=moe_config, 
    )
    inference_duration = time.time() - inference_start_time
    # print(f"Inference duration for {exp_name}: {round(inference_duration/3600, 2):.2f} hr)")
    # exp_output_dir = os.path.join(output_dir, exp_name)
    # embeddings_path = save_embeddings(
    #     embeddings=embeddings,
    #     individual_ids=individual_ids,
    #     index_dts=index_dts,
    #     output_path=exp_output_dir,
    #     model_name=exp_name,
    #     additional_metadata={
    #         'model_path': model_path,
    #         'model_type': model_type,
    #         'use_mixed_precision': use_mixed_precision,
    #     }
    # )
    add_info = ""
    safe_exp_name = exp_name.replace('-', '_').replace('.', '_') + add_info
    
    table_name = f"a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_{safe_exp_name}_{LOB}_embedding"
    bq_table_path = save_embeddings_to_bigquery(
        embeddings=embeddings,
        individual_ids=individual_ids,
        index_dts=index_dts,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        table_name=table_name,
        exp_name=exp_name,
        model_type=model_type,
        if_exists="replace"
    )
    results[exp_name] = {
        'bq_table_path': bq_table_path,
        # 'embeddings_path': embeddings_path,
        'embedding_shape': embeddings.shape,
        'model_type': model_type,
        'model_path': model_path,
        'inference_duration_hr': round(inference_duration / 3600, 2),
        'status': 'success'
    }

    # Free model memory
    del model
    del embeddings
    torch.cuda.empty_cache()

  0%|          | 0/1 [00:00<?, ?it/s]

Writing 2,886,355 rows to BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding
  Columns: 260 (embedding_dim=256)
  Estimated payload: 3.53 GB
  Chunking into 8 uploads of ~408,496 rows each
  ✓ Chunk 1/8: rows [0 – 408,496) uploaded
  ✓ Chunk 2/8: rows [408,496 – 816,992) uploaded
  ✓ Chunk 3/8: rows [816,992 – 1,225,488) uploaded
  ✓ Chunk 4/8: rows [1,225,488 – 1,633,984) uploaded
  ✓ Chunk 5/8: rows [1,633,984 – 2,042,480) uploaded
  ✓ Chunk 6/8: rows [2,042,480 – 2,450,976) uploaded
  ✓ Chunk 7/8: rows [2,450,976 – 2,859,472) uploaded
  ✓ Chunk 8/8: rows [2,859,472 – 2,886,355) uploaded
✅ Loaded 2,886,355 rows to edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding


In [128]:
embeddings.shape

(2886355, 512)

#### Replicate IP model pipeline

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, 
    average_precision_score, 
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)
from sklearn.base import clone
from sklearn.model_selection import train_test_split
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings('ignore')
import time
from dataclasses import dataclass

In [7]:
# constant
# EMBEDDING_BASE = "embedding_output/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2"
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"

# BigQuery embedding table pattern (matches save_embeddings_to_bigquery output)
# Template: a964286_te4exp_3lob_exp_round5_v2_{safe_exp_name}_commercial_all_sample_embedding
def get_commercial_embedding_table(exp_name: str, add_info: str = '') -> str:
    """Build the full BigQuery table ID for a commercial embedding experiment.
    edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_v2_exp2b_flash_learned_pool_asym_focalloss_commercial_all_sample_embedding
    edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_v2_exp2b_flash_learned_pool_asym_focalloss_densesampler_commercial_all_sample_embedding
    edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round10_v0_exp2b_flash_learned_pool_v2_commercial_embedding
    edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding
    edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding
    """
    safe_exp_name = exp_name.replace('-', '_').replace('.', '_') + add_info
    table_name = f"a964286_te4exp_3lob_exp_round10_v0_{safe_exp_name}_commercial_embedding"
    return f"{PROJECT_ID}.{DATASET_ID}.{table_name}"


EXPERIMENT_NAMES = [
    # 'exp1_dense_baseline_opt_config',
    # 'exp1_dense_baseline_pure_legacy', 
    # 'exp2b_flash_learned_pool', 
    # 'exp2b_flash_learned_pool_v4_asym_focalloss'
    # 'exp2b_flash_learned_pool_v5_asym_focalloss_dense_sampler'
    "exp2b_flash_learned_pool_round9_v3"
    # 'exp2b_flash_learned_pool_v2', 
    # 'exp6_auxiliary_free_v3'
    
    
]
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"
FEATURES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a964286_commercial_ip_heldout_transformer_matched_final_dataset_4_te_experiment_round5_downstream"
OOT_CUTOFF_DATE = "2023-10-16"
TARGET_COLUMN = "ip6"
EXCLUDE_COLUMNS = frozenset([
    # Keys and identifiers
    'individual_id', 'member_id', 'index_dt', 'birth_dt', 'feature_end_dt',
    
    # Outcome columns (target and related)
    'ip6', 'sum_ip6_admits', 'sum_ip6_los', 'sum_ip6_acu_days',
    
    # Eligibility/continuity flags
    'mon_3_include', 'mon_6_include', 'mon_12_include',
    'exclude_ip', 'include_post_6_status',
    
    # Split key
    'ind_id_last_digit',
    
    # Leakage columns (cost amounts, outreach flags from previous model)
    'clm_allowed_amt_1yr', 'clm_allowed_amt_2yr', 'clm_allowed_amt_3mo', 'clm_allowed_amt_6mo',
    'clm_paid_amt_1yr', 'clm_paid_amt_2yr', 'clm_paid_amt_3mo', 'clm_paid_amt_6mo',
    'clm_par_allowed_amt_1yr', 'clm_par_allowed_amt_2yr', 'clm_par_allowed_amt_3mo', 'clm_par_allowed_amt_6mo',
    'clm_par_paid_amt_1yr', 'clm_par_paid_amt_2yr', 'clm_par_paid_amt_3mo', 'clm_par_paid_amt_6mo',
    'clm_srv_copay_amt_1yr', 'clm_srv_copay_amt_3mo', 'clm_srv_copay_amt_6mo',
    'covid_19', 'hpd_major_flag', 'chronic',
    'txt_member', 'txt_referral', 'txt_1yr_outreach', 'talked'
])
# Data-level downsampling configuration
# Previous model used 10:1 negative sampling (table: yc_a565095_cp_ip_neg_10_trs_3)
NEGATIVE_DOWNSAMPLE_RATIO = 10  # Keep 10 negatives per 1 positive
APPLY_DOWNSAMPLING = True  # Set to False to disable downsampling

In [41]:
# Understand the time lapse between edp-prod-storage.edp_ent_sdoheir_cns.a834793_Commercial_final_dataset_4_te_experiment and edp-prod-storage.edp_ent_sdoheir_cns.a964286_commercial_heldout_transformer_input_4_te_experiment_round_5
# Row	discrepancy_bucket	member_count	pct_of_total	avg_diff_days	min_diff_days	max_diff_days
# 1	0: Exact match	6839956	97.53	0.0	0	0
# 2	2: 1 week - 1 month	4607	0.07	0.019101367484263269	-30	30
# 3	3: 1-3 months	21900	0.31	0.02187214611872133	-90	90
# 4	4: 3-6 months	45604	0.65	0.015261819138672406	-153	153
# 5	5: 6-12 months	100909	1.44	0.10750279955207313	-334	334

In [8]:
# =============================================================================
# METRIC FUNCTIONS
# =============================================================================

def lift_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """Calculate lift at top percentile. Lift = precision@k / baseline_prevalence."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    precision_at_k = y_true[top_k_indices].mean()
    baseline = y_true.mean()
    return precision_at_k / baseline if baseline > 0 else 0.0


def true_positives_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> int:
    """Count true positives in top percentile."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return int(y_true[top_k_indices].sum())


def precision_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """Calculate precision at top percentile."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return float(y_true[top_k_indices].mean())


def compute_split_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    """
    Compute all metrics for a single split.
    
    Returns:
        Dict with metric names as keys (without split prefix)
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    
    return {
        'auc_roc': roc_auc_score(y_true, y_prob),
        'auc_pr': average_precision_score(y_true, y_prob),
        'brier': brier_score_loss(y_true, y_prob),
        'lift_1pct': lift_at_percentage(y_true, y_prob, 0.01),
        'lift_5pct': lift_at_percentage(y_true, y_prob, 0.05),
        'lift_10pct': lift_at_percentage(y_true, y_prob, 0.10),
        'tp_1pct': true_positives_at_percentage(y_true, y_prob, 0.01),
        'precision_1pct': precision_at_percentage(y_true, y_prob, 0.01),
        'n_samples': len(y_true),
        'n_positives': int(y_true.sum()),
        'prevalence': float(y_true.mean()),
    }


In [9]:
# =============================================================================
# DATA PREPARATION FUNCTIONS
# =============================================================================
import glob

def load_embeddings_from_bigquery(
    table_id: str,
    project_id: Optional[str] = None,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Load embeddings from a BigQuery table.
    
    Returns a DataFrame with individual_id, index_dt, and embedding_0...embedding_N columns,
    matching the schema produced by save_embeddings_to_bigquery().
    
    Args:
        table_id: Full table ID (project.dataset.table)
        project_id: GCP project ID (optional if table_id is fully qualified)
        verbose: Print loading info
        
    Returns:
        DataFrame ready for join_embeddings_with_features()
    """
    client = bigquery.Client(project=project_id) if project_id else bigquery.Client()
    
    query = f"SELECT * FROM `{table_id}`"
    
    if verbose:
        print(f"  Loading embeddings from BigQuery: {table_id}")
    
    df = client.query(query).to_dataframe()
    embedding_cols = [c for c in df.columns if c.startswith('emb')]
    if '_' in embedding_cols[0]:
        embedding_cols = sorted(
            embedding_cols,
            key=lambda c: int(c.split('_')[1])
        )
    else:
        embedding_cols = sorted(
            embedding_cols,
            key=lambda c: int(c.replace('emb', ''))
        )
        rename_map = {c: f"embedding_{c.replace('emb', '')}" for c in embedding_cols}
        df.rename(columns=rename_map, inplace=True)
        embedding_cols = [rename_map[c] for c in embedding_cols]
    
    if verbose:
        print(f"  Loaded {len(df):,} rows with {len(embedding_cols)} embedding dimensions")
    
    df['index_dt'] = pd.to_datetime(df['index_dt']).dt.strftime('%Y-%m-%d')
    
    keep_cols = ['individual_id', 'index_dt'] + embedding_cols
    return df[keep_cols]


def load_embeddings_from_dir(embedding_path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load embeddings from the NPZ file for a given experiment.
    
    Args:
        exp_name: Experiment name (e.g., 'exp1_dense_baseline')
        base_dir: Base directory containing experiment subdirectories
        
    Returns:
        embeddings: numpy array [num_members, 256]
        individual_ids: list of member IDs
        index_dts: list of index dates
    """
    if os.path.isdir(embedding_path):
        npz_files = glob.glob(os.path.join(embedding_path, "embeddings_*.npz"))
        if not npz_files:
            raise FileNotFoundError(f"No NPZ files found in {embedding_path}")
        npz_path = sorted(npz_files)[-1]  # Use most recent
    elif '*' in embedding_path:
        npz_files = glob.glob(embedding_path)
        if not npz_files:
            raise FileNotFoundError(f"No files matching {embedding_path}")
        npz_path = sorted(npz_files)[-1]
    else:
        npz_path = embedding_path
    
    data = np.load(npz_path, allow_pickle=True)
    
    return (
        data['embeddings'],
        data['individual_ids'],
        data['index_dts']
    )

def load_mbrs_have_embed(embedding_path: str):
    import pandas as pd
    if os.path.isdir(embedding_path):
        csv_files = glob.glob(os.path.join(embedding_path, "embeddings_*.csv"))
        if not csv_files:
            raise FileNotFoundError(f"No embedding ID files found in {embedding_path}")
        csv_file = sorted(csv_files)[-1]
        return pd.read_csv(csv_file)
    
    
def create_embedding_df(embeddings: np.ndarray,
        individual_ids: np.ndarray,
        index_dts: np.ndarray
    ) -> pd.DataFrame:
    """
    Create a DataFrame from embedding data.
    col: individual_id, index_dt, embedding_0...embedding_255
    """
    embedding_dim = embeddings.shape[1]
    embedding_cols = [f'embedding_{i}' for i in range(embedding_dim)]
    
    df = pd.DataFrame({
        'individual_id': individual_ids,
        'index_dt': pd.to_datetime(index_dts).strftime('%Y-%m-%d')
    })
    
    embedding_df = pd.DataFrame(embeddings, columns=embedding_cols)
    return pd.concat([df, embedding_df], axis=1)

def join_embeddings_with_features(
    emb_df: pd.DataFrame,
    df_features: pd.DataFrame
) -> pd.DataFrame:
    """
    Join embeddings with features on (individual_id, index_dt).
    Applies eligibility filters automatically.
    """
    # Standardize date format
    df_features = df_features.copy()
    df_features['index_dt'] = pd.to_datetime(df_features['index_dt']).dt.strftime('%Y-%m-%d')
    emb_df['index_dt'] = pd.to_datetime(emb_df['index_dt']).dt.strftime('%Y-%m-%d')
    emb_df['individual_id'] = emb_df['individual_id'].astype(str)
    df_features['individual_id'] = df_features['individual_id'].astype(str)
    # Inner join
    df_merged = df_features.merge(
        emb_df,
        on=['individual_id', 'index_dt'],
        how='inner'
    )
    
    # # mon_6_include filter is done in sql and the other two columsn do not present
    # if 'mon_6_include' in df_merged.columns:
    #     df_merged = df_merged[df_merged['mon_6_include'] == 1]
    # if 'exclude_ip' in df_merged.columns:
    #     df_merged = df_merged[(df_merged['exclude_ip'] == 0) | (df_merged['exclude_ip'].isna())]
    # if 'include_post_6_status' in df_merged.columns:
    #     df_merged = df_merged[df_merged['include_post_6_status'] == 1]
    
    # Remove duplicates
    df_merged = df_merged.drop_duplicates(
        subset=['individual_id', 'index_dt'], 
        keep='last'
    )
    
    return df_merged

def create_data_splits(
    df: pd.DataFrame,
    oot_cutoff_date: str = OOT_CUTOFF_DATE
) -> Dict[str, pd.DataFrame]:
    """
    Create train/val/test/OOT splits based on ind_id_last_digit and date.
    
    Split Logic (matching previous pipeline with improved consistency):
        - Train: ind_id_last_digit 0-7 AND date <= cutoff
        - Val:   ind_id_last_digit 8 AND date <= cutoff
        - Test:  ind_id_last_digit 9 AND date <= cutoff
        - OOT:   date > cutoff (all ind_id_last_digit values)
        - OOT_strict: date > cutoff AND ind_id_last_digit 9
    
    Note: OOT may contain members also present in train/val/test at earlier dates.
    This tests temporal generalization (model performance on future time periods).
    
    Returns:
        Dict with keys 'train', 'val', 'test', 'oot', 'oot_strict'
    """
    df = df.copy()
    df['_index_dt_parsed'] = pd.to_datetime(df['index_dt'])
    
    oot_cutoff = pd.to_datetime(oot_cutoff_date)
    
    splits = {
        'train': df[(df['ind_id_last_digit'].isin([0,1,2,3,4,5,6,7]))&(df['_index_dt_parsed'] <= oot_cutoff)],
        'val': df[(df['ind_id_last_digit'] == 8) & (df['_index_dt_parsed'] <= oot_cutoff)],
        'test': df[(df['ind_id_last_digit'] == 9) & (df['_index_dt_parsed'] <= oot_cutoff)],
        'oot': df[df['_index_dt_parsed'] > oot_cutoff],
        'oot_strict': df[(df['_index_dt_parsed'] > oot_cutoff) & (df['ind_id_last_digit'] == 9)]
    }
    print("Data splits created:")
    for name, split_df in splits.items():
        if len(split_df) > 0:
            prevalence = split_df[TARGET_COLUMN].mean() * 100
            print(f"  {name}: {len(split_df):,} rows, {int(split_df[TARGET_COLUMN].sum()):,} positives ({prevalence:.2f}%)")
        else:
            print(f"  {name}: EMPTY")
            
    # Remove temp column
    for key in splits:
        splits[key] = splits[key].drop(columns=['_index_dt_parsed'])
    
    return splits

def identify_feature_columns(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Identify embedding and tabular feature columns.
    
    Returns:
        Tuple of (embedding_features, tabular_features)
    """
    all_cols = set(df.columns)
    
    embedding_features = sorted([c for c in all_cols if c.startswith('embedding_')])
    
    excluded = EXCLUDE_COLUMNS | set(embedding_features) | {'_exp_name', 'index_dt_parsed', '_index_dt_parsed'}
    tabular_features = sorted([
        c for c in all_cols 
        if c not in excluded and c != TARGET_COLUMN
    ])
    
    return embedding_features, tabular_features

def downsample_negatives(
    X: pd.DataFrame, 
    y: pd.Series,
    ratio: int = NEGATIVE_DOWNSAMPLE_RATIO,
    random_state: int = 42
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Downsample negative class to achieve target ratio (matching previous pipeline).
    
    The previous model used a pre-sampled table (yc_a565095_cp_ip_neg_10_trs_3)
    with approximately 10:1 negative-to-positive ratio. This function replicates
    that data-level rebalancing strategy.
    
    Args:
        X: Feature DataFrame
        y: Target Series (0/1)
        ratio: Number of negatives to keep per positive (default: 10)
        random_state: Random seed for reproducibility
        
    Returns:
        Tuple of (X_resampled, y_resampled)
        
    Example:
        If y has 1000 positives and 50000 negatives with ratio=10:
        - Keep all 1000 positives
        - Randomly sample 10000 negatives (10 * 1000)
        - Return resampled data with ~10:1 ratio
    """
    np.random.seed(random_state)
    
    # Separate positive and negative indices
    pos_mask = y == 1
    neg_mask = y == 0
    
    pos_indices = X.index[pos_mask].tolist()
    neg_indices = X.index[neg_mask].tolist()
    
    n_positives = len(pos_indices)
    n_negatives = len(neg_indices)
    target_n_negatives = int(n_positives * ratio)
    
    # If we already have fewer negatives than target, keep all
    if n_negatives <= target_n_negatives:
        print(f"  Downsampling: No action needed (current ratio: {n_negatives/n_positives:.1f}:1)")
        return X, y
    
    # Randomly sample negatives
    sampled_neg_indices = np.random.choice(neg_indices, size=target_n_negatives, replace=False)
    
    # Combine indices
    keep_indices = pos_indices + sampled_neg_indices.tolist()
    
    X_resampled = X.loc[keep_indices].copy()
    y_resampled = y.loc[keep_indices].copy()
    
    # Shuffle to mix positives and negatives
    shuffle_idx = np.random.permutation(len(X_resampled))
    X_resampled = X_resampled.iloc[shuffle_idx].reset_index(drop=True)
    y_resampled = y_resampled.iloc[shuffle_idx].reset_index(drop=True)
    
    print(f"  Downsampling: {n_negatives}:{n_positives} ({n_negatives/n_positives:.1f}:1) → "
          f"{target_n_negatives}:{n_positives} ({ratio}:1)")
    
    return X_resampled, y_resampled


def prepare_features(
    df: pd.DataFrame,
    feature_cols: List[str]
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Prepare feature matrix X and target y, handling missing values.
    """
    X = df[feature_cols].copy()
    y = df[TARGET_COLUMN].astype(int)
    
    # Fill missing values
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    X[numeric_cols] = X[numeric_cols].fillna(0)
    
    cat_cols = X.select_dtypes(include=['object', 'category']).columns
    X[cat_cols] = X[cat_cols].fillna('missing')
    
    return X, y



In [10]:
from dataclasses import dataclass
@dataclass
class PreparedData:
    """Container for prepared evaluation data. Prepare once, evaluate many models."""
    X_splits: Dict[str, pd.DataFrame]
    y_splits: Dict[str, pd.Series]
    feature_cols: List[str]
    embedding_features: List[str]
    tabular_features: List[str]
    cat_feature_indices: List[int]  # Column indices for CatBoost
    feature_set: str
    embedding_path: str
    downsampled: bool = True  # Whether training data was downsampled

def prepare_evaluation_data(
    df_features: pd.DataFrame,
    embedding_location_path: str = "",
    feature_set: str = 'embedding_only',
    oot_cutoff_date: str = OOT_CUTOFF_DATE,
    downsample_ratio: Optional[float] = None,
    random_state: int = 42
) -> PreparedData:
    """
    Prepare data once for multiple model evaluations.
    
    This function decouples data preparation from model training, allowing
    the same prepared data to be used across multiple models efficiently.
    
    Args:
        df_features: DataFrame with features and outcomes (from BigQuery)
        embedding_location_path: BigQuery table ID (project.dataset.table) or local path to NPZ dir.
                                 BigQuery tables are auto-detected by the '.' separator convention.
                                 Not needed for tabular_only.
        feature_set: One of 'embedding_only', 'tabular_only', 'hybrid'
        oot_cutoff_date: Date string for OOT split cutoff
        downsample_ratio: If provided, downsample training negatives to this ratio 
                          (e.g., 10.0 for 10:1 negative:positive ratio). 
                          Only applied to training set. Set to 10.0 to match previous model.
        random_state: Random seed for downsampling reproducibility
        
    Returns:
        PreparedData object containing X_splits, y_splits, and metadata
    """
    total_start_time = time.time()
    step_times = {}
    
    # Validate feature_set
    valid_feature_sets = {'embedding_only', 'tabular_only', 'hybrid'}
    if feature_set not in valid_feature_sets:
        raise ValueError(f"feature_set must be one of {valid_feature_sets}")
    
    step_start = time.time()
    print(f"\n Loading and preparing data...")
    # Load embeddings (skip for tabular_only)
    if feature_set != 'tabular_only':
        is_bigquery = embedding_location_path.count('.') >= 2
        print(f"  Loading embeddings from {'BigQuery' if is_bigquery else 'local'}: {embedding_location_path}")
        
        if is_bigquery:
            emb_df = load_embeddings_from_bigquery(embedding_location_path)
        else:
            embeddings, individual_ids, index_dts = load_embeddings_from_dir(embedding_location_path)
            print(f"  Creating embedding DataFrame...")
            emb_df = create_embedding_df(embeddings, individual_ids, index_dts)
        
        print(f"  Joining embeddings with features...")
        if feature_set == 'embedding_only':
            df_merged = join_embeddings_with_features(emb_df, df_features[['individual_id', 'index_dt', TARGET_COLUMN, 'ind_id_last_digit']])
        else:
            df_merged = join_embeddings_with_features(emb_df, df_features)
    else:
        if embedding_location_path:
            is_bigquery = embedding_location_path.count('.') >= 2
            if is_bigquery:
                print(f"  Preparing tabular-only data scoped to members in BigQuery embedding table...")
                emb_df = load_embeddings_from_bigquery(embedding_location_path)
                df_members = emb_df[['individual_id', 'index_dt']]
            else:
                print(f"  Preparing tabular-only data (no embeddings) joined with member ID in embedding table...")
                df_members = load_mbrs_have_embed(embedding_location_path)
            df_merged = join_embeddings_with_features(df_members[['individual_id', 'index_dt']], df_features)
        # using entire table without joining to match 30% samples
        else:
            # For tabular-only, just apply filters directly
            df_merged = df_features.copy()
            df_merged['index_dt'] = pd.to_datetime(df_merged['index_dt']).dt.strftime('%Y-%m-%d')
            if 'mon_6_include' in df_merged.columns:
                df_merged = df_merged[df_merged['mon_6_include'] == 1]
            if 'exclude_ip' in df_merged.columns:
                df_merged = df_merged[(df_merged['exclude_ip'] == 0) | (df_merged['exclude_ip'].isna())]
            if 'include_post_6_status' in df_merged.columns:
                df_merged = df_merged[df_merged['include_post_6_status'] == 1]
            df_merged = df_merged.drop_duplicates(subset=['individual_id', 'index_dt'], keep='last')    
            
    step_times['step1_data_loading'] = time.time() - step_start
    print(f"  Step 1 complete ({step_times['step1_data_loading']:.2f}s)")
    print(f"\n[Step 2/6] Creating data splits...")
    # Create splits
    step_start = time.time()
    splits = create_data_splits(df_merged)
    step_times['step2_create_splits'] = time.time() - step_start
    print(f"  Step 2 complete ({step_times['step2_create_splits']:.2f}s)")
    
    # Identify feature columns
    embedding_features, tabular_features = identify_feature_columns(df_merged)

    
    # Select features based on feature_set
    if feature_set == 'embedding_only':
        feature_cols = embedding_features
    elif feature_set == 'tabular_only':
        feature_cols = tabular_features
    else:  # hybrid
        feature_cols = tabular_features + embedding_features
    
    step_start = time.time()
    print(f"\n[Step 4/6] Preparing feature matrices for each split...")
        
    # Prepare feature matrices for each split
    X_splits, y_splits = {}, {}
    for split_name, split_df in splits.items():
        if len(split_df) > 0:
            X_splits[split_name], y_splits[split_name] = prepare_features(split_df, feature_cols)
            
    step_times['step4_prepare_features'] = time.time() - step_start
    print(f"  Step 4 complete ({step_times['step4_prepare_features']:.2f}s)")
    
   
    # Apply downsampling to training set only (if requested)
    downsampled = False
    if downsample_ratio is not None and 'train' in X_splits:
        step_start = time.time()
        print(f"\n[Step 5/6] Rebalance the training dataset with a ratio of {downsample_ratio}...") 
        X_splits['train'], y_splits['train'] = downsample_negatives(
            X_splits['train'], 
            y_splits['train'], 
            ratio=downsample_ratio,
            random_state=random_state
        )
        downsampled = True    
    
    print(f"Finish data preparation, total time: {time.time() - total_start_time}")
    # Pre-compute categorical column indices for CatBoost (only for tabular/hybrid)
    cat_feature_indices = []
    if feature_set != 'embedding_only' and 'train' in X_splits:
        cat_cols = X_splits['train'].select_dtypes(include=['object', 'category']).columns
        cat_feature_indices = [X_splits['train'].columns.get_loc(c) for c in cat_cols]
    
    return PreparedData(
        X_splits=X_splits,
        y_splits=y_splits,
        feature_cols=feature_cols,
        embedding_features=embedding_features,
        tabular_features=tabular_features,
        cat_feature_indices=cat_feature_indices,
        feature_set=feature_set,
        embedding_path=embedding_location_path,
        downsampled=downsampled
    )

In [11]:
# =============================================================================
# MODEL EVALUATION FUNCTION
# =============================================================================
def evaluate_model_on_splits(
    model,
    X_splits: Dict[str, pd.DataFrame],
    y_splits: Dict[str, pd.Series],
    apply_scaling: bool = False,
    cat_feature_indices: Optional[List[int]] = None
) -> Dict[str, Dict[str, float]]:
    """
    Train model on train split, evaluate on all splits.
    
    Args:
        model: sklearn-compatible model with fit() and predict_proba()
        X_splits: Dict with 'train', 'val', 'test', 'oot' DataFrames
        y_splits: Dict with corresponding target Series
        apply_scaling: Whether to apply StandardScaler
        cat_feature_indices: List of categorical column indices (for CatBoost)
    
    Returns:
        Dict with split names as keys, metrics dict as values
    """
    total_start_time = time.time()
    step_times = {}
    
    # Clone model to avoid modifying original
    model = clone(model)
    
    X_train, y_train = X_splits['train'], y_splits['train']
    
    step_start = time.time()

    # Handle scaling
    scaler = None
    if apply_scaling:
        scaler = StandardScaler()
        X_train_processed = scaler.fit_transform(X_train)
        step_times['scaling'] = time.time() - step_start
        print(f"\n Scaling done {step_times['scaling']}...")
    else:
        X_train_processed = X_train

    
    # Handle CatBoost-specific training
    model_type = type(model).__name__
    
    step_start = time.time()
    if model_type == 'CatBoostClassifier':
        # CatBoost uses Pool for categorical features
        from catboost import Pool
        cat_indices = cat_feature_indices if cat_feature_indices else []
        train_pool = Pool(X_train, y_train, cat_features=cat_indices)
        val_pool = Pool(X_splits['val'], y_splits['val'], cat_features=cat_indices)
        
        model.fit(train_pool, eval_set=val_pool, verbose=0)
    else:
        model.fit(X_train_processed, y_train)
    step_times['model_fit'] = time.time() - step_start
    print(f"\n Fit model done {model_type}: {step_times['model_fit']}")
    
    # Evaluate on all splits
    step_start = time.time()
    results = {}
    for split_name in tqdm(['val', 'test', 'oot', 'oot_strict']):
        print(f"\n Evaluating {split_name}...")
        X_split = X_splits.get(split_name)
        y_split = y_splits.get(split_name)
        
        if X_split is None or len(X_split) == 0:
            continue
        
        # Apply same preprocessing
        if apply_scaling and scaler is not None:
            X_processed = scaler.transform(X_split)
        else:
            X_processed = X_split
        
        # Predict
        if model_type == 'CatBoostClassifier' and cat_feature_indices:
            from catboost import Pool
            pool = Pool(X_split, cat_features=cat_feature_indices)
            y_prob = model.predict_proba(pool)[:, 1]
        else:
            y_prob = model.predict_proba(X_processed)[:, 1]
        
        # Compute metrics
        results[split_name] = compute_split_metrics(np.array(y_split), y_prob)
    step_times['model_eval'] = time.time() - step_start
    print(f"\n Evaluate model done {model_type}: {step_times['model_eval']}")
    return results

In [12]:
def evaluate_with_prepared_data(
    prepared_data: PreparedData,
    ml_model_object: Any,
    exp_name: str,
    apply_scaling: bool = False
) -> Dict[str, Any]:
    """
    Evaluate a model using pre-prepared data.
    
    This is the efficient way to evaluate multiple models on the same dataset.
    Call prepare_evaluation_data() once, then call this function for each model.
    
    Args:
        prepared_data: PreparedData object from prepare_evaluation_data()
        ml_model_object: Pre-configured sklearn-compatible model
        exp_name: Experiment name for result identification
        apply_scaling: Whether to apply StandardScaler (True for LR, False for tree-based)
    
    Returns:
        Dict with exp_name, model_type, feature_set, and all metrics
    """
    # Determine if we should use categorical features
    use_cat_features = (
        prepared_data.feature_set != 'embedding_only' and 
        len(prepared_data.cat_feature_indices) > 0
    )
    
    # Evaluate model
    split_results = evaluate_model_on_splits(
        model=ml_model_object,
        X_splits=prepared_data.X_splits,
        y_splits=prepared_data.y_splits,
        apply_scaling=apply_scaling,
        cat_feature_indices=prepared_data.cat_feature_indices if use_cat_features else None
    )
    
    # Build output dictionary
    output = {
        'exp_name': exp_name,
        'model_type': type(ml_model_object).__name__,
        'feature_set': prepared_data.feature_set,
        'n_features': len(prepared_data.feature_cols),
    }
    
    # Flatten split results with prefixes
    for split_name, metrics in split_results.items():
        for metric_name, value in metrics.items():
            output[f'{split_name}_{metric_name}'] = value
    
    return output


def evaluate_all_experiments(
    experiment_configs: List[Dict],
    df_features: pd.DataFrame,
    downsample_ratio: Optional[float] = None
) -> pd.DataFrame:
    """
    Evaluate multiple experiments efficiently by grouping by data requirements.
    
    Data is prepared once per unique (embedding_path, feature_set, downsample_ratio) 
    combination, then reused for all models in that group.
    
    Args:
        experiment_configs: List of dicts, each with:
            - embedding_location_path: str
            - ml_model_object: model
            - exp_name: str
            - feature_set: str (optional, default 'embedding_only')
            - apply_scaling: bool (optional, default False)
            - downsample_ratio: float (optional, overrides global downsample_ratio)
        df_features: DataFrame with features and outcomes
        downsample_ratio: Global downsample ratio for all experiments.
                          Use 10.0 to match previous model's 10:1 negative sampling.
                          Can be overridden per-experiment in config.
    
    Returns:
        DataFrame with one row per experiment, all metrics as columns
    """
    from collections import defaultdict
    
    # Group configs by (embedding_path, feature_set, downsample_ratio) to avoid redundant data preparation
    groups = defaultdict(list)

    for config in tqdm(experiment_configs):
        embedding_path = config.get('embedding_location_path', '')
        feature_set = config.get('feature_set', 'embedding_only')
        # Per-experiment downsample_ratio overrides global
        ds_ratio = config.get('downsample_ratio', downsample_ratio)
        # Use tuple as key for grouping (include downsample_ratio since it affects data)
        key = (embedding_path, feature_set, ds_ratio)
        groups[key].append(config)
    
    results = []
    prepared_cache = {}  # Cache prepared data for reuse  
    for (embedding_path, feature_set, ds_ratio), group_configs in tqdm(groups.items()):
        print(f"==========================================")
        # Prepare data once for this group
        cache_key = (embedding_path, feature_set, ds_ratio)
        if cache_key not in prepared_cache:
            ds_str = f", downsample={ds_ratio}:1" if ds_ratio else ""
            print(f"Preparing data for: feature_set={feature_set}{ds_str}, path={embedding_path[:50] if embedding_path else 'N/A'}...")
            prepared_cache[cache_key] = prepare_evaluation_data(
                df_features=df_features,
                embedding_location_path=embedding_path,
                feature_set=feature_set,
                downsample_ratio=ds_ratio
            )
        
        prepared_data = prepared_cache[cache_key]
        
        # Evaluate each model in this group using prepared data
        for config in group_configs:
            model = config['ml_model_object']
            exp_name = config['exp_name']
            apply_scaling = config.get('apply_scaling', False)
            
            print(f"  Evaluating: {exp_name} ({type(model).__name__})")
            
            result = evaluate_with_prepared_data(
                prepared_data=prepared_data,
                ml_model_object=model,
                exp_name=exp_name,
                apply_scaling=apply_scaling
            )
            results.append(result)
    
    return pd.DataFrame(results)

In [13]:
lr_model = LogisticRegression(
    max_iter=2000, 
    solver='lbfgs', 
    class_weight='balanced',
    random_state=42
)
catboost_model = CatBoostClassifier(
    iterations=2500,
    depth=7,
    learning_rate=0.025,
    grow_policy='SymmetricTree',
    auto_class_weights='Balanced',
    od_wait=80,
    use_best_model=True,
    random_seed=42,
    verbose=0
)
# Match previous model's best configuration
catboost_model_legacy = CatBoostClassifier(
    iterations=2436,
    depth=7,
    learning_rate=0.027,  # Rounded from 0.026766501358942353
    random_strength=3,
    l2_leaf_reg=2.95,
    border_count=136,
    min_data_in_leaf=30,
    grow_policy='SymmetricTree',
    od_wait=84,
    bootstrap_type='Bernoulli',
    subsample=0.79,
    leaf_estimation_iterations=8,
    loss_function='Logloss',
    eval_metric='AUC',
    od_type='Iter',
    use_best_model=True,
    random_seed=42,
    thread_count=-1,
    verbose=0
)

In [22]:
# embedding_path = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_v2_exp2b_flash_learned_pool_asym_focalloss_commercial_all_sample_embedding'
# embedding_path = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_v2_exp2b_flash_learned_pool_asym_focalloss_densesampler_commercial_all_sample_embedding'
# embedding_path = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round7_512emb_exp2b_flash_learned_poolweight200_commercial_all_sample_embedding'
# embedding_path = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round6_6_8M_exp2b_flash_learned_poolweight200_commercial_all_sample_embedding'
# embedding_path = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round10_v0_exp2b_flash_learned_pool_v2_commercial_embedding'
# embedding_path = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding'
embedding_path = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding'
experiment_configs = [
    # These 3 share (exp1 path, embedding_only) - data prepared ONCE
    # {
    #     'embedding_location_path': f"{EMBEDDING_BASE}/exp1_dense_baseline_pure_legacy",
    #     'ml_model_object': catboost_model,
    #     'exp_name': "exp1_legacy_catboost_emb_only",
    #     'feature_set': 'embedding_only',
    #     'apply_scaling': False
    # },
    {
        # 'embedding_location_path': f"{EMBEDDING_BASE}/exp2b_flash_learned_pool", # exp_round6
        'embedding_location_path': embedding_path,
        'ml_model_object': catboost_model,
        'exp_name': "exp1_legacy_dbcheck_round5_3lobs_1-5M_catboost_emb_only",
        'feature_set': 'embedding_only',
        'apply_scaling': False
    },
    # {
    #     # 'embedding_location_path': f"{EMBEDDING_BASE}/exp2b_flash_learned_pool", # exp_round6
    #     'embedding_location_path': embedding_path,
    #     'ml_model_object': catboost_model,
    #     'exp_name': "exp2b_round9_v3_3lobs_1-5M_catboost_emb_only",
    #     'feature_set': 'embedding_only',
    #     'apply_scaling': False
    # },
    # {
    #     'embedding_location_path': f"{EMBEDDING_BASE}/exp2b_flash_learned_pool_v2",
    #     'ml_model_object': catboost_model,
    #     'exp_name': "exp2b_catboost_emb_only",
    #     'feature_set': 'embedding_only',
    #     'apply_scaling': False
    # },
    # {
    #     'embedding_location_path': f"{EMBEDDING_BASE}/exp6_auxiliary_free_v3",
    #     'ml_model_object': catboost_model,
    #     'exp_name': "exp6_catboost_emb_only",
    #     'feature_set': 'embedding_only',
    #     'apply_scaling': False
    # },
    # # These 3 share (exp1 path, embedding_only) - join with embedding taboe
    # {
    #     'embedding_location_path': f"{EMBEDDING_BASE}/exp1_dense_baseline_pure_legacy",
    #     'ml_model_object': catboost_model,
    #     'exp_name': "tabular_only_catboost",
    #     'feature_set': 'tabular_only',
    #     'apply_scaling': False
    # },
    # {
    #     'embedding_location_path': f"{EMBEDDING_BASE}/exp1_dense_baseline_pure_legacy",
    #     'ml_model_object': catboost_model,
    #     'exp_name': "exp1_legacy_catboost_hybrid",
    #     'feature_set': 'hybrid',
    #     'apply_scaling': False
    # },
    {
        # 'embedding_location_path': f"{EMBEDDING_BASE}/exp2b_flash_learned_pool",
        'embedding_location_path': embedding_path,
        'ml_model_object': catboost_model,
        'exp_name': "exp1_legacy_dbcheck_round5_3lobs_1-5M_catboost_hybrid",
        'feature_set': 'hybrid',
        'apply_scaling': False
    },
    # {
    #     'embedding_location_path': f"{EMBEDDING_BASE}/exp2b_flash_learned_pool_v2",
    #     'ml_model_object': catboost_model,
    #     'exp_name': "exp2b_catboost_hybrid",
    #     'feature_set': 'hybrid',
    #     'apply_scaling': False
    # },
    # {
    #     'embedding_location_path': f"{EMBEDDING_BASE}/exp6_auxiliary_free_v3",
    #     'ml_model_object': catboost_model,
    #     'exp_name': "exp6_catboost_hybrid",
    #     'feature_set': 'hybrid',
    #     'apply_scaling': False
    # }, 
    # {
    #     'embedding_location_path': f"", # use full dataset commericial what is the ceiline
    #     'ml_model_object': catboost_model,
    #     'exp_name': "full_tabular_only_catboost",
    #     'feature_set': 'tabular_only',
    #     'apply_scaling': False
    # }

]

##### import feature tables

In [23]:
import google.auth
from google.auth import impersonated_credentials
from google.cloud import bigquery
client = bigquery.Client()
credentials, project_id= google.auth.default()
print('credentials:', credentials, ', project:', project)
import pandas as pd
from tqdm.notebook import tqdm
client = bigquery.Client()


credentials: <google.oauth2.credentials.Credentials object at 0x7f0ce3b5e290> , project: edp-prod-css-sdoh


In [16]:
feature_sql = f"""
SELECT *
FROM `{FEATURES_TABLE}`
"""

In [17]:
df_ip_features = client.query(feature_sql).to_dataframe()

In [24]:
df_ip_features['ip6'].value_counts()

ip6
0    4245904
1      38690
Name: count, dtype: Int64

In [19]:
df_ip_features['index_dt'] = pd.to_datetime(df_ip_features['index_dt']).dt.strftime('%Y-%m-%d')
# embedding_dfs['exp1_dense_baseline']['index_dt'] = pd.to_datetime(embedding_dfs['exp1_dense_baseline']['index_dt']).dt.strftime('%Y-%m-%d')

#### Feature importance

In [38]:
# =============================================================================
# COMMERCIAL: SHAP Feature Importance Analysis
# =============================================================================
# Demonstrates the additional value of embeddings via SHAP
# Uses the hybrid feature set to see embedding vs tabular importance

embedding_path_shap = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round10_v0_exp2b_flash_learned_pool_v2_commercial_embedding'

prepared_hybrid_commercial = prepare_evaluation_data(
    df_features=df_ip_features,
    embedding_location_path=embedding_path_shap,
    feature_set='hybrid',
    downsample_ratio=10.0
)

# Train CatBoost on hybrid and capture the fitted model
from sklearn.base import clone as sk_clone

catboost_shap = sk_clone(catboost_model)
cat_indices = prepared_hybrid_commercial.cat_feature_indices if prepared_hybrid_commercial.cat_feature_indices else []
from catboost import Pool
train_pool_shap = Pool(
    prepared_hybrid_commercial.X_splits['train'],
    prepared_hybrid_commercial.y_splits['train'],
    cat_features=cat_indices,
)
val_pool_shap = Pool(
    prepared_hybrid_commercial.X_splits['val'],
    prepared_hybrid_commercial.y_splits['val'],
    cat_features=cat_indices,
)
catboost_shap.fit(train_pool_shap, eval_set=val_pool_shap, verbose=0)
print("Commercial CatBoost (hybrid) trained for SHAP analysis")


 Loading and preparing data...
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round10_v0_exp2b_flash_learned_pool_v2_commercial_embedding
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round10_v0_exp2b_flash_learned_pool_v2_commercial_embedding
  Loaded 2,886,355 rows with 256 embedding dimensions
  Joining embeddings with features...
  Step 1 complete (137.43s)

[Step 2/6] Creating data splits...
Data splits created:
  train: 843,297 rows, 7,505 positives (0.89%)
  val: 105,272 rows, 946 positives (0.90%)
  test: 105,458 rows, 936 positives (0.89%)
  oot: 758,134 rows, 7,026 positives (0.93%)
  oot_strict: 75,490 rows, 718 positives (0.95%)
  Step 2 complete (27.00s)

[Step 4/6] Preparing feature matrices for each split...
  Step 4 complete (24.81s)

[Step 5/6] Rebalance the training dataset with a ratio of 10.0...
  Downsampling: 835792:7505 (111.4:1) → 75050:7505 (10.0:1)
Finish data

In [40]:
# Run SHAP
commercial_shap_df, commercial_proportion_df = compute_shap_feature_importance(
    fitted_model=catboost_shap,
    X_eval=prepared_hybrid_commercial.X_splits['test'],
    feature_cols=prepared_hybrid_commercial.feature_cols,
    embedding_features=prepared_hybrid_commercial.embedding_features,
    top_k_list=[10, 20, 50],
    max_samples=5000,
    model_name="commercial_catboost_hybrid",
    verbose=True,
)



SHAP FEATURE IMPORTANCE: commercial_catboost_hybrid
  Sampled 5000 rows from 105458 for SHAP computation

  Top 20 features by mean |SHAP|:
 rank                   feature  mean_abs_shap  is_embedding
    1               embedding_9       0.155555          True
    2             embedding_106       0.106008          True
    3             embedding_255       0.084980          True
    4             embedding_209       0.081658          True
    5 cerebrovascular_condition       0.071708         False
    6              embedding_61       0.066106          True
    7                       age       0.064458         False
    8             embedding_181       0.063310          True
    9              embedding_85       0.062141          True
   10             embedding_134       0.061822          True
   11        uniq_dx_cd_cnt_1yr       0.058135         False
   12              embedding_51       0.055190          True
   13              embedding_24       0.054625          True
   14

#### Evaluate on performance

In [27]:
results_df_exp_round5_legacy_dbcheck_epoch3 = evaluate_all_experiments(experiment_configs, df_ip_features, downsample_ratio=10.0)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Preparing data for: feature_set=embedding_only, downsample=10.0:1, path=edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4ex...

 Loading and preparing data...
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding
  Loaded 2,862,079 rows with 256 embedding dimensions
  Joining embeddings with features...
  Step 1 complete (126.81s)

[Step 2/6] Creating data splits...
Data splits created:
  train: 843,553 rows, 7,623 positives (0.90%)
  val: 105,386 rows, 940 positives (0.89%)
  test: 105,446 rows, 959 positives (0.91%)
  oot: 758,134 rows, 7,026 positives (0.93%)
  oot_strict: 75,490 rows, 718 positives (0.95%)
  Step 2 complete (5.94s)

[Step 4/6] Preparing feature matrices for each split...
  Step 4 complete (10.17s)

[Step 5/6] Rebalan

  0%|          | 0/4 [00:00<?, ?it/s]


 Evaluating val...

 Evaluating test...

 Evaluating oot...

 Evaluating oot_strict...

 Evaluate model done CatBoostClassifier: 1.444206953048706
Preparing data for: feature_set=hybrid, downsample=10.0:1, path=edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4ex...

 Loading and preparing data...
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding
  Loaded 2,862,079 rows with 256 embedding dimensions
  Joining embeddings with features...
  Step 1 complete (136.49s)

[Step 2/6] Creating data splits...
Data splits created:
  train: 843,553 rows, 7,623 positives (0.90%)
  val: 105,386 rows, 940 positives (0.89%)
  test: 105,446 rows, 959 positives (0.91%)
  oot: 758,134 rows, 7,026 positives (0.93%)
  oot_strict: 75,490 rows, 718 positiv

  0%|          | 0/4 [00:00<?, ?it/s]


 Evaluating val...

 Evaluating test...

 Evaluating oot...

 Evaluating oot_strict...

 Evaluate model done CatBoostClassifier: 5.033761739730835


In [28]:
results_df_exp_round5_legacy_dbcheck_epoch3.T

,0,1
exp_name,exp1_legacy_dbcheck_round5_3lobs_1-5M_catboost...,exp1_legacy_dbcheck_round5_3lobs_1-5M_catboost...
model_type,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,hybrid
n_features,256,789
val_auc_roc,0.692165,0.791827
val_auc_pr,0.032854,0.071569
val_brier,0.103475,0.090557
val_lift_1pct,8.304649,16.289889
val_lift_5pct,4.298117,7.191899
val_lift_10pct,3.29806,4.808784


In [26]:
results_df_exp_round5_legacy_dbcheck_epoch3.T

,0,1
exp_name,exp1_legacy_dbcheck_round5_3lobs_1-5M_catboost...,exp1_legacy_dbcheck_round5_3lobs_1-5M_catboost...
model_type,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,hybrid
n_features,256,789
val_auc_roc,0.714637,0.812743
val_auc_pr,0.037338,0.078795
val_brier,0.103212,0.092211
val_lift_1pct,9.066571,17.269658
val_lift_5pct,4.898403,7.574183
val_lift_10pct,3.624905,5.113706


In [49]:
results_df_exp_round9 = evaluate_all_experiments(experiment_configs, df_ip_features, downsample_ratio=10.0)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Preparing data for: feature_set=embedding_only, downsample=10.0:1, path=edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4ex...

 Loading and preparing data...
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding
  Loaded 2,886,355 rows with 256 embedding dimensions
  Joining embeddings with features...
  Step 1 complete (191.40s)

[Step 2/6] Creating data splits...
Data splits created:
  train: 843,297 rows, 7,505 positives (0.89%)
  val: 105,272 rows, 946 positives (0.90%)
  test: 105,458 rows, 936 positives (0.89%)
  oot: 758,134 rows, 7,026 positives (0.93%)
  oot_strict: 75,490 rows, 718 positives (0.95%)
  Step 2 complete (6.23s)

[Step 4/

  0%|          | 0/4 [00:00<?, ?it/s]


 Evaluating val...

 Evaluating test...

 Evaluating oot...

 Evaluating oot_strict...

 Evaluate model done CatBoostClassifier: 1.5128114223480225
Preparing data for: feature_set=hybrid, downsample=10.0:1, path=edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4ex...

 Loading and preparing data...
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding
  Loading embeddings from BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round9_3lobs_1-5M_decoupled_training_cooccur_embedding_exp2b_flash_learned_pool_v2_commercial_embedding
  Loaded 2,886,355 rows with 256 embedding dimensions
  Joining embeddings with features...
  Step 1 complete (146.22s)

[Step 2/6] Creating data splits...
Data splits created:
  train: 843,297 rows, 7,505 positives (0.89%)
  val: 105,272 rows, 946 positives (0.90%)
  test: 105,458 rows, 936 positi

  0%|          | 0/4 [00:00<?, ?it/s]


 Evaluating val...

 Evaluating test...

 Evaluating oot...

 Evaluating oot_strict...

 Evaluate model done CatBoostClassifier: 5.397356271743774


In [50]:
results_df_exp_round9.T

,0,1
exp_name,exp2b_round9_v3_3lobs_1-5M_catboost_emb_only,exp2b_round9_v3_3lobs_1-5M_catboost_hybrid
model_type,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,hybrid
n_features,256,789
val_auc_roc,0.777906,0.809717
val_auc_pr,0.057518,0.077385
val_brier,0.085989,0.082722
val_lift_1pct,13.222574,16.607553
val_lift_5pct,6.681523,7.463853
val_lift_10pct,4.566683,5.031808


In [34]:
results_df_exp_round10.T

,0,1
exp_name,exp_round10_3lobs_11M_catboost_emb_only,exp_round10_3lobs_11M_v3catboost_hybrid
model_type,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,hybrid
n_features,256,789
val_auc_roc,0.784325,0.811385
val_auc_pr,0.068359,0.081523
val_brier,0.084672,0.083326
val_lift_1pct,14.491941,16.607553
val_lift_5pct,6.766099,7.104404
val_lift_10pct,4.492686,5.000095


In [52]:
results_df_exp_round6.T

,0,1
exp_name,exp_round6_3lobs_6-8M_pretrain_multi_gpu_test_...,exp_round6_3lobs_6-8M_pretrain_multi_gpu_test_...
model_type,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,hybrid
n_features,256,789
val_auc_roc,0.792723,0.818982
val_auc_pr,0.067724,0.089577
val_brier,0.086462,0.086261
val_lift_1pct,14.51535,17.936825
val_lift_5pct,6.798895,8.187694
val_lift_10pct,4.818882,5.264499


In [27]:
results_df_exp_round7.T

,0,1
exp_name,exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_...,exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_...
model_type,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,hybrid
n_features,512,1045
val_auc_roc,0.788394,0.815027
val_auc_pr,0.062301,0.080113
val_brier,0.086108,0.082968
val_lift_1pct,13.561972,16.825003
val_lift_5pct,6.545212,7.890957
val_lift_10pct,4.760629,5.097034


In [168]:
results_df.T

,0,1,2,3,4,5,6
exp_name,exp1_catboost_emb_only,exp2b_catboost_emb_only,exp6_catboost_emb_only,tabular_only_catboost,exp1_catboost_hybrid,exp2b_catboost_hybrid,exp6_catboost_hybrid
model_type,CatBoostClassifier,CatBoostClassifier,CatBoostClassifier,CatBoostClassifier,CatBoostClassifier,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,embedding_only,embedding_only,tabular_only,hybrid,hybrid,hybrid
n_features,256,256,256,533,789,789,789
val_auc_roc,0.705162,0.76454,0.767553,0.82302,0.811607,0.808269,0.806389
val_auc_pr,0.033984,0.068763,0.06157,0.089957,0.085453,0.088302,0.086262
val_brier,0.103071,0.092434,0.092617,0.122659,0.089124,0.088832,0.089673
val_lift_1pct,8.383835,14.807812,14.807812,19.163051,18.400884,18.836408,18.183122
val_lift_5pct,4.374527,6.398562,6.681492,8.183195,8.009084,8.09614,8.074376
val_lift_10pct,3.395155,4.50511,4.537756,5.212434,5.125379,5.070969,5.19067


In [75]:
results_df_fulltabular.T

,0,1
exp_name,exp1_opt_catboost_emb_only,exp1_opt_catboost_hybrid
model_type,CatBoostClassifier,CatBoostClassifier
feature_set,embedding_only,hybrid
n_features,256,789
val_auc_roc,0.774795,0.814289
val_auc_pr,0.064216,0.090694
val_brier,0.085718,0.083468
val_lift_1pct,14.55299,19.614899
val_lift_5pct,6.891596,8.008582
val_lift_10pct,4.604935,5.195041


In [58]:
results_df_exp_round5.T.to_excel("experiment_logs/exp_round5_3lob_1-5M_1epoch_128batch_dim256_commercial_ip_downstream_eval_asym_focalloss.xlsx")

In [88]:
results_df_exp_round5.T.to_excel("experiment_logs/exp_round5_3lob_1-5M_1epoch_128batch_dim256_commercial_ip_downstream_eval_asym_focalloss_densesampler.xlsx")

In [106]:
df_commercial_downstream = pd.read_excel("experiment_logs/commercial_ip_1-5M_30pctsample_downstream.xlsx")

In [109]:
df_commercial_downstream.columns

Index(['exp_name', 'full_tabular_only_catboost',
       'embedding_matched_tabular_only_catboost', 'exp1_catboost_emb_only',
       'exp1_opt_catboost_emb_only', 'exp2b_catboost_emb_only',
       'exp_round5_exp2b_v4_asym_focalloss_catboost_emb_only',
       'exp_round5_exp2b_v4_asym_focalloss_densesampler_catboost_emb_only',
       'exp_round6_exp2b_catboost_emb_only', 'exp6_catboost_emb_only',
       'exp1_catboost_hybrid', 'exp1_opt_catboost_hybrid',
       'exp2b_catboost_hybrid', 'exp6_catboost_hybrid',
       'exp_round6-3.4M_exp2b_catboost_hybrid',
       'exp_round5_exp2b_v4_asym_focalloss_densesampler_catboost_hybrid',
       'exp_round5_exp2b_v4_asym_focalloss_catboost_hybrid'],
      dtype='object')

### Medicare embedding generation

In [54]:
import google.auth
from google.auth import impersonated_credentials
from google.cloud import bigquery
client = bigquery.Client()
credentials, project= google.auth.default()
print('credentials:', credentials, ', project:', project)
import pandas as pd
from tqdm.notebook import tqdm
client = bigquery.Client()

credentials: <google.oauth2.credentials.Credentials object at 0x7f1a6012ba60> , project: edp-prod-css-sdoh


In [55]:
# import members not in the trainingset of the transformer
# Didn't sampled and 30% generate all 2.78M
medicare_sql_code = """
select * from edp-prod-storage.edp_ent_sdoheir_cns.a834793_Medicare_holdout_members_with_features
"""
df_me = client.query(medicare_sql_code).to_dataframe()

In [56]:
df_me['lob'] = 'Medicare'

In [57]:
# sample before and after 2023-10-16 (post part will be used for oot validation)
# 0.3 samples for efficent evaluations of embeddings
df_me['index_dt'] = pd.to_datetime(df_me['index_dt'])
df_me_b4_oct = df_me[df_me['index_dt'] <= pd.to_datetime("2023-10-16")]
df_me_after_oct = df_me[df_me['index_dt'] > pd.to_datetime("2023-10-16")]
df_me_b4_oct_sample = df_me_b4_oct.sample(frac=0.3, random_state=42)
df_me_after_oct_sample = df_me_after_oct.sample(frac=0.3, random_state=42)
df_me_sample = pd.concat([df_me_b4_oct_sample,
                         df_me_after_oct_sample])

In [60]:
df_me_sample.head()

,individual_id,member_id,index_dt,feature_end_dt,business_ln_cd,ip6,sum_ip6_admits,sum_ip6_los,mon_6_include,camemhpd_aff,...,camemmbrshp_agenbr,camemmbrshp_age65_74,e_caperetdem444,e_caperetdem21220,gender_cd,age_in_months,cd,target,dt_cnt,lob
2396840,1110185,10145331450098,2023-01-16,2022-10-18,ME,0,0,0,1,0,...,60,0,1387832,38866,0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*...,711*712*713*715*720*721*721*721*722*722*723*72...,"1,14,1582,10309,10490,62979,74402,74516*1192*1...","948*948*948*1,14,1000,1802,1828,5327,5647,5688...",43,Medicare
2072684,8215261072,10136591640098,2023-02-16,2022-11-18,ME,1,2,12,1,1,...,78,0,2400330,89193,0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*...,934*934*934*934*934*935*935*935*935*936*936*93...,"1,14,1592,12809,13204,54744,74112,74358,74516,...","1,14,999,1480,1713,1997,5204,5530,5688*955*927...",65,Medicare
1060080,5790487386188,10133786950098,2023-04-16,2023-01-16,ME,0,0,0,1,0,...,67,1,3904104,118296,0*0*0*0,802*808*809*809,"1,14,1592,26346,55035,55038,74358,74514,74516,...","1,14,1048,1466,2428,5058,5080,5098,5620,5688*1...",4,Medicare
557490,393134900551,10123259320098,2023-05-16,2023-02-15,ME,0,0,0,1,0,...,66,1,6576514,178467,1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*1*...,783*783*783*784*784*784*784*785*785*785*785*78...,"1,14,1592,26427,55294,56156,58045,74303,75133,...","908*903*1,14,1010,1363,5058,5098,5203,5606,592...",59,Medicare
930931,8549744686,10120459690098,2023-05-16,2023-02-15,ME,0,0,0,1,0,...,75,0,2411786,74008,0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*0*...,881*882*883*883*884*884*884*885*886*886*887*88...,"1438*1141*1073*982,1192,1438*953*1191*900,1246...","940*930*913,948,979*908*948*894,955*940*930*90...",56,Medicare


In [59]:
del df_me, df_me_b4_oct, df_me_after_oct, df_me_b4_oct_sample, df_me_after_oct_sample

In [68]:
MODEL_PATHS = {
#     # Experiment 1: Dense Baseline (no Flash Attention, no MoE)
#     'exp1_dense_baseline_pure_legacy': 
#         'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_pure_legacy/saved_models/'
#         'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs128_ep1_d256_20251230_055716_final.pt', 
    
#     # Experiment 1b: Dense Baseline (same opt config as 2b and 6)
#     'exp1_dense_baseline_opt_config': 
#         'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_opt_config/saved_models/'
#         'exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs64_ep1_d256_20260108_183616_final.pt',

#     # Experiment 2b: Flash Attention + Learned Pooling (no MoE)
#     'exp2b_flash_learned_pool_v2': 
#         'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp2b_flash_learned_pool_v2/saved_models/'
#         'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20251230_114137_final.pt',

    
    # Experiment 2b: Flash Attention + Learned Pooling (no MoE) + (focalloss / focalloss_dense_sampler)
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/'
    # 'exp2b_flash_learned_pool_v4_asym_focalloss/saved_models/'
    # 'exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260221_120056_final.pt'
    
#     # Experiment 2: exp round 7 with 512 dimensions; regular BCE with weights 200
    'exp2b_flash_learned_pool': 
    'logs/exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim/'
    'exp2b_flash_learned_poolexp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final/saved_models/'
    'exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final.pt',
    
    # Experiment 2: exp round 6 with 256 dimensions; regular BCE with weights 200
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2/'
    # 'exp2b_flash_learned_pool_6-8M/saved_models/'
    # 'exp_round6_3lobs_6-8M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260304_221738_final.pt'
     
#     # Experiment 6: Flash + MoE with DeepSeek auxiliary-free balancing
#     'exp6_auxiliary_free_v3': 
#         'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/'
#         'exp6_auxiliary_free_v3/saved_models/'
#         'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp6_auxiliary_free_bs128_ep1_d256_20251231_152438_final.pt',
    
    # Round 6; 
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2/'
    # 'exp2b_flash_learned_pool/saved_models/'
    # 'exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260110_112709_final.pt'
}

In [69]:
import time

results = {}
batch_size = 64
output_dir = "embedding_output/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2"
# output_dir = "embedding_output/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2"
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"
LOB = 'medicare'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for exp_name, model_path in tqdm(MODEL_PATHS.items()):
    cleanup_gpu_memory(verbose=False)
    model, config, moe_config, use_mixed_precision, model_type = load_model_from_checkpoint(
        model_path=MODEL_PATHS[exp_name],
        device=device,
        verbose=True
    )
    
    inference_start_time = time.time()
    embeddings, individual_ids, index_dts = generate_embeddings(
        model=model,
        config=config,
        data=df_me_sample,
        device=device,
        id_column='individual_id',  # Commercial uses individual_id
        lob_value=None,              # Medicare data already has lob column
        desc_prefix='Mecicare',
        batch_size=batch_size,
        use_mixed_precision=use_mixed_precision,
        verbose=True,
        multi_gpu=True,           
        moe_config=moe_config, 
    )
    inference_duration = time.time() - inference_start_time
    print(f"Inference duration for {exp_name}: {round(inference_duration/3600, 2):.2f} hr)")
    exp_output_dir = os.path.join(output_dir, exp_name)
    # embeddings_path = save_embeddings(
    #     embeddings=embeddings,
    #     individual_ids=individual_ids,
    #     index_dts=index_dts,
    #     output_path=exp_output_dir,
    #     model_name=exp_name,
    #     additional_metadata={
    #         'model_path': model_path,
    #         'model_type': model_type,
    #         'use_mixed_precision': use_mixed_precision,
    #     }
    # )
    # add_info = "_asym_focalloss"
    # safe_exp_name = exp_name.replace('-', '_').replace('.', '_') + add_info
    # table_name = f"a964286_te4exp_3lob_exp_round5_v2_{safe_exp_name}_{LOB}_all_sample_embedding"

    add_info = ""
    safe_exp_name = exp_name.replace('-', '_').replace('.', '_') + add_info
    table_name = f"a964286_te4exp_3lob_exp_round7_512dim_{safe_exp_name}_{LOB}_all_sample_embedding"        
    bq_table_path = save_embeddings_to_bigquery(
        embeddings=embeddings,
        individual_ids=individual_ids,
        index_dts=index_dts,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        table_name=table_name,
        exp_name=exp_name,
        model_type=model_type,
        if_exists="replace"
    )
    results[exp_name] = {
        'embeddings_path': bq_table_path,
        'embedding_shape': embeddings.shape,
        'model_type': model_type,
        'model_path': model_path,
        'inference_duration_hr': round(inference_duration/3600, 2),
        'status': 'success'
    }

    # Free model memory
    del model
    del embeddings
    torch.cuda.empty_cache()

  0%|          | 0/1 [00:00<?, ?it/s]


Loading model from: logs/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2/exp2b_flash_learned_pool_6-8M/saved_models/exp_round6_3lobs_6-8M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260304_221738_final.pt
  Model type: FlashAttentionTransformer
  Embedding size: 256
  N layers: 6
  Use learned attention pooling: True
✅ Model loaded successfully!
   Total parameters: 25,325,209
   Mixed precision: True
   Device: cuda


MECICARE EMBEDDING GENERATION
Samples: 836,844 | Batch: 64 | GPUs: 4
Workers: 4 | Mixed precision: True
ID column: individual_id
Multi-GPU mode: 4 GPUs
  Cloning model to GPU 0...
  Cloning model to GPU 1...
  Cloning model to GPU 2...
  Cloning model to GPU 3...
  GPU 0: samples 0 to 209,211 (209,211 samples)
  GPU 1: samples 209,211 to 418,422 (209,211 samples)
  GPU 2: samples 418,422 to 627,633 (209,211 samples)
  GPU 3: samples 627,633 to 836,844 (209,211 samples)


Generating Mecicare embeddings (4 GPUs):   0%|          | 0/836844 [00:00<?, ?it/s]

LazyClinicalDataset initialized with 209,211 samples (lazy loading)
LazyClinicalDataset initialized with 209,211 samples (lazy loading)
LazyClinicalDataset initialized with 209,211 samples (lazy loading)
LazyClinicalDataset initialized with 209,211 samples (lazy loading)

✅ Complete! Time: 2924.3s | Speed: 286 samples/s
   Effective: 1,145 samples/s (across 4 GPUs)
   Output: (836844, 256)
Inference duration for exp2b_flash_learned_pool: 0.81 hr)
Writing 836,844 rows to BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round7_512dim_exp2b_flash_learned_pool_medicare_all_sample_embedding
  Columns: 260 (embedding_dim=256)
  Estimated payload: 1.02 GB
  Chunking into 3 uploads of ~408,496 rows each
  ✓ Chunk 1/3: rows [0 – 408,496) uploaded
  ✓ Chunk 2/3: rows [408,496 – 816,992) uploaded
  ✓ Chunk 3/3: rows [816,992 – 836,844) uploaded
✅ Loaded 836,844 rows to edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round7_512dim_exp2b_flash_learned_pool_medicar

In [110]:
# =============================================================================
# MEDICARE: Embedding Generation for Round 10, 9, and 7 Models
# =============================================================================

MODEL_PATHS_NEW_MEDICARE = {
    'exp_round10_formal_exp2b_flash_learned_pool_d256':
        'logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/'
        'exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt',

    'exp_round7_exp2b_flash_learned_pool_d512':
        'logs/exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim/'
        'exp2b_flash_learned_poolexp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final/'
        'saved_models/'
        'exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final.pt',

    'exp_round9_decoupled_exp2b_flash_learned_pool_v2_d256':
        'logs/exp_round9_3lobs_1-5M_decoupled_training_embedding_v4_256dim/exp2b_flash_learned_pool_v2/saved_models/'
        'exp_round9_3lobs_1-5M_decoupled_training_embedding_v4_256dim_exp2b_flash_learned_pool_bs128_ep1_d256_20260310_123547_final.pt',
}

results_new_medicare = {}
batch_size = 64
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"
LOB = 'medicare'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for exp_name, model_path in tqdm(MODEL_PATHS_NEW_MEDICARE.items(), desc="Medicare embedding generation"):
    cleanup_gpu_memory(verbose=False)
    model, config, moe_config, use_mixed_precision, model_type = load_model_from_checkpoint(
        model_path=model_path,
        device=device,
        verbose=True
    )

    inference_start_time = time.time()
    embeddings, individual_ids, index_dts = generate_embeddings(
        model=model,
        config=config,
        data=df_me_sample,
        device=device,
        id_column='individual_id',
        lob_value=None,
        desc_prefix='Medicare',
        batch_size=batch_size,
        use_mixed_precision=use_mixed_precision,
        verbose=True,
        multi_gpu=True,
        moe_config=moe_config,
    )
    inference_duration = time.time() - inference_start_time
    print(f"Inference duration for {exp_name}: {round(inference_duration/3600, 2):.2f} hr")

    safe_exp_name = exp_name.replace('-', '_').replace('.', '_')
    table_name = f"a964286_te4exp_3lob_{safe_exp_name}_{LOB}_all_sample_embedding"
    bq_table_path = save_embeddings_to_bigquery(
        embeddings=embeddings,
        individual_ids=individual_ids,
        index_dts=index_dts,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        table_name=table_name,
        exp_name=exp_name,
        model_type=model_type,
        if_exists="replace"
    )
    results_new_medicare[exp_name] = {
        'bq_table_path': bq_table_path,
        'embedding_shape': embeddings.shape,
        'model_type': model_type,
        'model_path': model_path,
        'inference_duration_hr': round(inference_duration / 3600, 2),
        'status': 'success'
    }

    del model
    del embeddings
    torch.cuda.empty_cache()

print("\n=== Medicare Embedding Generation Summary ===")
for exp_name, result in results_new_medicare.items():
    print(f"  {exp_name}: {result['embedding_shape']} -> {result['bq_table_path']}")

The scikit-learn version is 1.7.2


### Medicaid downstream

In [53]:
import os
import sys
import glob
import time
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Union
from dataclasses import dataclass
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm import tqdm

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.base import clone
from catboost import CatBoostClassifier, Pool

# BigQuery
import google.auth
from google.cloud import bigquery


#### Configs

In [88]:
# BigQuery Tables - CORRECTED
PROJECT_ID = "edp-prod-storage"
DATASET_ID = "edp_ent_sdoheir_cns"

# =============================================================================
# HELDOUT TABLES (members NOT in TE pretraining 10% sample)
# These are used for downstream evaluation to avoid data leakage
# =============================================================================
HELDOUT_FEATURES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a964286_medicaid_ip_heldout_non_embedding_features"
HELDOUT_OUTCOME_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a964286_medicaid_ip_heldout_outcome_ip"
HELDOUT_TE_INPUT_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a964286_medicaid_ip_heldout_te_inference_input"

# =============================================================================
# FULL DATASET TABLES (all members, including those in pretrain)
# Only use these for reference/comparison, NOT for downstream evaluation
# =============================================================================
FULL_FEATURES_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a964286_medicaid_ip_final_dataset_4_te_experiment_2023_non_embedding_features"
FULL_OUTCOME_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a964286_medicaid_ip_final_dataset_4_te_experiment_2023_outcome_ip"

# =============================================================================
# TE CROSSWALK TABLES (for ID mapping between formats)
# =============================================================================
MEMBER_CROSSWALK_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a834793_Medicaid_member_train_ending"
TE_SEQUENCE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a834793_Medicaid_o3_train_ending"
PRETRAIN_10PCT_TABLE = f"{PROJECT_ID}.{DATASET_ID}.a834793_Combined_All_LOB_o3_train_10pct_sample"

# Legacy table references (from Eric Ma's original pipeline - for reference only)
# These are from the anbc-hcb-dev project, not used in current downstream eval
# LEGACY_FEATURES_TABLE = "anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_non_embedding_features"
# LEGACY_EMBEDDINGS_TABLE = "anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_embeddings"
# LEGACY_OUTCOME_TABLE = "anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_outcome_ip"

# Default to HELDOUT tables for downstream evaluation
FEATURES_TABLE = HELDOUT_FEATURES_TABLE
OUTCOME_TABLE = HELDOUT_OUTCOME_TABLE

# Note: Embeddings will come from transformer inference on HELDOUT_TE_INPUT_TABLE
# No pre-computed embeddings table for heldout - we generate them fresh

# Target variable
TARGET_COLUMN = "acute_ip_flag"

# Member ID column (primary key for joining)
MEMBER_KEY = "asdb_member_key"

# Random seeds for reproducibility (matches original Eric Ma pipeline)
RANDOM_STATE = 35  # For train/test split (matches Eric's train_test_split random_state)
UNDERSAMPLE_RANDOM_STATE = 53  # For undersampling (Eric uses 53 for RandomUnderSampler)
CATBOOST_RANDOM_SEED = 53  # For CatBoost model (Eric uses 53 for random_seed)

# Class imbalance handling
# Original finding: CatBoost works better with 0.2 undersampling ratio
CATBOOST_UNDERSAMPLE_RATIO = 0.2  # 20% minority-to-majority (5:1)
XGBOOST_UNDERSAMPLE_RATIO = 0.03  # 3% minority-to-majority (~33:1)

# Train/Val/Test split ratios (original: 80/10/10)
TRAIN_SIZE = 0.8
VAL_SIZE = 0.1
TEST_SIZE = 0.1

# =============================================================================
# OUT-OF-TIME (OOT) VALIDATION CONFIGURATION
# =============================================================================
# For time-based train/test split similar to commercial IP:
# - Data before cutoff: used for train/val/test (stratified random split)
# - Data after cutoff: used for OOT (out-of-time) validation
# This tests temporal generalization of the model.
#
# Note: Commercial IP uses ind_id_last_digit for deterministic splitting.
# Medicaid doesn't have this column, so we use stratified random split
# for train/val/test, matching Eric's original Medicaid IP pipeline.
OOT_CUTOFF_DATE = "2023-10-16"  # Same as commercial IP for consistency

# Sampling fraction for efficient embedding generation (optional)
# Set to None to use full data, or 0.3 for 30% sample like commercial
EMBEDDING_SAMPLE_FRAC = None  # Use full data for Medicaid heldout (already ~90% of total)

# =============================================================================
# CATBOOST TUNED HYPERPARAMETERS
# =============================================================================
# From Optuna optimization in original pipeline (optuna_results_catboost.csv)
# Best trial 42: AUC = 0.8737 (from optuna_catboost.log)
# Note: Eric's final model uses params from lines 513-521 of catboost.py
CATBOOST_TUNED_PARAMS = {
    'learning_rate': 0.015742881221129403,
    'iterations': 2665,
    'l2_leaf_reg': 0.222046549398224,
    'depth': 7,
    'random_seed': CATBOOST_RANDOM_SEED,  # Eric uses 53 for CatBoost
    'verbose': 0,
    'thread_count': -1,  # Use all available threads (-1), Eric used 15
    'use_best_model': True,
    # Note: Original didn't use auto_class_weights, relied on undersampling instead
}

# Alternative: Balanced class weights model (without undersampling)
CATBOOST_BALANCED_PARAMS = {
    'iterations': 2500,
    'depth': 7,
    'learning_rate': 0.025,
    'grow_policy': 'SymmetricTree',
    'auto_class_weights': 'Balanced',
    'od_wait': 80,
    'use_best_model': True,
    'random_seed': CATBOOST_RANDOM_SEED,  # Use same seed as tuned params
    'verbose': 0,
    'thread_count': -1,
}

# =============================================================================
# SELECTED FEATURES FROM RFECV
# =============================================================================
# These 243 non-embedding features were selected by RFECV in the original pipeline
# The full list (499) includes these + 256 embedding features (emb0-emb255)

SELECTED_TABULAR_FEATURES = [
    # COA Population Group (categorical)
    'coa_population_group',
    
    # ED Visits - Year 1
    'sum_ed_visits_yr1', 'ed_flag_yr1', 'sum_avoidable_yr1', 'sum_unnecessary_yr1',
    'sum_preventable_yr1', 'low_sev_ed_visits_yr1', 'low_med_sev_ed_visits_yr1',
    'med_sev_ed_visits_yr1', 'med_high_sev_ed_visits_yr1', 'high_sev_ed_visits_yr1',
    'high_sev_ed_flag_yr1',
    
    # ED Visits - Year 2
    'sum_ed_visits_yr2', 'sum_avoidable_yr2', 'sum_preventable_yr2',
    'med_sev_ed_visits_yr2', 'med_high_sev_ed_visits_yr2', 'high_sev_ed_visits_yr2',
    
    # IP Admits
    'sum_acute_ip_admits_yr1', 'sum_acute_calc_los_yr1',
    'sum_acute_ip_admits_yr2', 'sum_acute_calc_los_yr2',
    
    # OP Visits
    'sum_op_visits_yr1', 'sum_op_visits_yr2',
    
    # EMIS Claims - Year 1
    'emis_community_clm_yr1', 'emis_ed_clm_yr1', 'emis_hh_clm_yr1', 'emis_home_clm_yr1',
    'emis_ip_clm_yr1', 'emis_ins_clm_yr1', 'emis_lab_clm_yr1', 'emis_mrx_clm_yr1',
    'emis_mh_clm_yr1', 'emis_misc_clm_yr1', 'emis_pcp_clm_yr1', 'emis_radio_clm_yr1',
    'emis_ambul_clm_yr1', 'emis_spec_clm_yr1',
    
    # LTC and COE Claims - Year 1
    'ltc_clm_yr1', 'coe_ip_hos_clm_yr1', 'coe_ip_non_hos_clm_yr1', 'coe_lab_clm_yr1',
    'coe_ltc_community_clm_yr1', 'coe_ltc_home_clm_yr1', 'coe_ltc_ins_clm_yr1',
    'coe_other_clm_yr1', 'coe_op_hos_clm_yr1', 'coe_op_non_hos_clm_yr1',
    'coe_anesth_clm_yr1', 'coe_eval_clm_yr1', 'coe_maternity_clm_yr1',
    'coe_mrx_clm_yr1', 'coe_mh_clm_yr1', 'coe_phy_clm_yr1', 'coe_surg_clm_yr1',
    'coe_radio_clm_yr1', 'uc_clm_yr1', 'obs_clm_yr1',
    
    # EMIS Claims - Year 2
    'emis_community_clm_yr2', 'emis_ed_clm_yr2', 'emis_hh_clm_yr2', 'emis_home_clm_yr2',
    'emis_ip_clm_yr2', 'emis_ins_clm_yr2', 'emis_lab_clm_yr2', 'emis_mrx_clm_yr2',
    'emis_mh_clm_yr2', 'emis_misc_clm_yr2', 'emis_pcp_clm_yr2', 'emis_radio_clm_yr2',
    'emis_ambul_clm_yr2', 'emis_spec_clm_yr2',
    
    # LTC and COE Claims - Year 2
    'ltc_clm_yr2', 'coe_ip_hos_clm_yr2', 'coe_ip_non_hos_clm_yr2', 'coe_lab_clm_yr2',
    'coe_other_clm_yr2', 'coe_op_hos_clm_yr2', 'coe_op_non_hos_clm_yr2',
    'coe_anesth_clm_yr2', 'coe_eval_clm_yr2', 'coe_maternity_clm_yr2',
    'coe_mrx_clm_yr2', 'coe_mh_clm_yr2', 'coe_phy_clm_yr2', 'coe_surg_clm_yr2',
    'coe_radio_clm_yr2', 'uc_clm_yr2', 'obs_clm_yr2',
    
    # Chronic Conditions (binary flags)
    'IDA', 'ANX', 'OST', 'AST', 'CHO', 'burns', 'CBD', 'CHF', 'CRF', 'CHD',
    'COP', 'DIA', 'esrd', 'EPL', 'CRO', 'MOH', 'HepC', 'HYP', 'HYC',
    'meta_cancer', 'liver_dis', 'MSS', 'OBE', 'oud', 'paralysis', 'hmd',
    'PVD', 'autoimmune', 'SCA', 'spinal_inj', 'back', 'substance', 'ALC', 'psychoses',
    'major_chronic_cnt',
    
    # Pharmacy Features - Year 1
    'rx_claim_cnt_yr1', 'days_supply_sum_yr1', 'ndc_cnt_yr1', 'gpi_cnt_yr1',
    'gpi4_cnt_yr1', 'gpi2_cnt_yr1', 'retail_fills_yr1', 'mail_order_fills_yr1',
    'generic_fills_yr1', 'branded_generic_fills_yr1', 'ss_brand_fills_yr1',
    'ms_brand_fills_yr1', 'formulary_fills_yr1', 'maint_drug_fills_yr1',
    'antidiabetic_scripts_yr1', 'antidiabetic_days_supply_yr1',
    'beta_blocker_scripts_yr1', 'beta_blocker_days_supply_yr1',
    'antihypertensive_scripts_yr1', 'antihypertensive_days_supply_yr1',
    'lipid_lowering_scripts_yr1', 'lipid_lowering_days_supply_yr1',
    'calcium_channel_blk_scripts_yr1', 'calcium_channel_blk_days_supply_yr1',
    'diuretic_scripts_yr1', 'diuretic_days_supply_yr1',
    'antianginal_agent_scripts_yr1', 'antianginal_agent_days_supply_yr1',
    'antidepressant_scripts_yr1', 'antidepressant_days_supply_yr1',
    'antipsychotic_scripts_yr1', 'antipsychotic_days_supply_yr1',
    'antianxiety_days_supply_yr1', 'anticonvulsant_scripts_yr1',
    'anticonvulsant_days_supply_yr1', 'inhaled_steroid_scripts_yr1',
    'inhaled_steroid_days_supply_yr1',
    
    # Pharmacy Features - Year 2
    'rx_claim_cnt_yr2', 'days_supply_sum_yr2', 'ndc_cnt_yr2', 'gpi_cnt_yr2',
    'gpi4_cnt_yr2', 'gpi2_cnt_yr2', 'retail_fills_yr2', 'generic_fills_yr2',
    'branded_generic_fills_yr2', 'ss_brand_fills_yr2', 'ms_brand_fills_yr2',
    'formulary_fills_yr2', 'maint_drug_fills_yr2',
    'antidiabetic_scripts_yr2', 'antidiabetic_days_supply_yr2',
    'beta_blocker_scripts_yr2', 'beta_blocker_days_supply_yr2',
    'antihypertensive_scripts_yr2', 'antihypertensive_days_supply_yr2',
    'lipid_lowering_scripts_yr2', 'lipid_lowering_days_supply_yr2',
    'calcium_channel_blk_scripts_yr2', 'calcium_channel_blk_days_supply_yr2',
    'diuretic_days_supply_yr2', 'antianginal_agent_scripts_yr2',
    'antidepressant_scripts_yr2', 'antidepressant_days_supply_yr2',
    'antipsychotic_scripts_yr2', 'antipsychotic_days_supply_yr2',
    'antianxiety_scripts_yr2', 'antianxiety_days_supply_yr2',
    'anticonvulsant_scripts_yr2', 'anticonvulsant_days_supply_yr2',
    'inhaled_steroid_days_supply_yr2',
    
    # Demographics
    'agenbr', 'gender', 'ethnicity_code', 'primarylanguage_desc',
    'tenure_yr1', 'tenure_yr2', 'urbsubr',
    
    # SDOH Scores
    'zip_weight_avg_medinc', 'acs_social_risk_score', 'sdi_score', 'svi_score',
    'adi_score', 'citizenship_index', 'education_index', 'food_access',
    'health_access', 'health_habits', 'housing_desert', 'housing_ownership',
    'housing_quality', 'income_index', 'income_inequality', 'language_score',
    'natural_disaster', 'poverty_score', 'proactive_health', 'racial_diversity',
    'social_isolation', 'technology_access', 'transport_access',
    'unemployment_index', 'water_quality', 'disability_score', 'health_infra',
    'csdi_social_risk_score',
    
    # Healthcare Utilization
    'sum_pcp', 'sum_spec', 'sum_ob', 'sum_dme', 'sum_chol_lab', 'sum_a1c_lab',
    'sum_chemo',
    
    # CMS Screening Flags
    'cms_alc_scrn', 'cms_col_scrn', 'cms_hepb_scrn', 'cms_nutrition',
    'cms_sti_scrn', 'cms_mam_scrn',
]

# Embedding features (256 dimensions)
EMBEDDING_FEATURES = [f'embedding_{i}' for i in range(256)]

# Categorical features requiring special handling
CATEGORICAL_FEATURES = [
    'coa_population_group', 'gender', 'ethnicity_code',
    'primarylanguage_desc', 'urbsubr',
    'cms_alc_scrn', 'cms_col_scrn', 'cms_hepb_scrn', 'cms_nutrition',
    'cms_sti_scrn', 'cms_mam_scrn',
]


In [89]:
MODEL_PATHS = {
    # Experiment 1: Dense Baseline (no Flash Attention, no MoE)
    'exp1_dense_baseline_pure_legacy': 
        'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_pure_legacy/saved_models/'
        'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs128_ep1_d256_20251230_055716_final.pt', 
    
    # Experiment 1b: Dense Baseline (same opt config as 2b and 6)
    'exp1_dense_baseline_opt_config': 
        'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_opt_config/saved_models/'
        'exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs64_ep1_d256_20260108_183616_final.pt',

    # Experiment 2b: Flash Attention + Learned Pooling (no MoE)
    'exp2b_flash_learned_pool_v2': 
        'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp2b_flash_learned_pool_v2/saved_models/'
        'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20251230_114137_final.pt',
    
    # Experiment 6: Flash + MoE with DeepSeek auxiliary-free balancing
    'exp6_auxiliary_free_v3': 
        'logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/'
        'exp6_auxiliary_free_v3/saved_models/'
        'exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp6_auxiliary_free_bs128_ep1_d256_20251231_152438_final.pt',
    
    # Round 6; 
    # 'exp2b_flash_learned_pool': 
    # 'logs/exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2/'
    # 'exp2b_flash_learned_pool/saved_models/'
    # 'exp_round6_3lobs_3-4M_pretrain_multi_gpu_test_v2_exp2b_flash_learned_pool_bs128_ep1_d256_20260110_112709_final.pt'
}

#### Eval metrics

In [90]:

# =============================================================================
# EVALUATION METRICS FUNCTIONS
# =============================================================================
# These match the original Medicaid IP model evaluation exactly

def lift_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """
    Calculate lift at top percentile.
    
    Lift = precision@k / baseline_prevalence
    
    This is the primary metric for the Medicaid IP model, measuring how
    many times better the model is at identifying positives in the top k%.
    
    Args:
        y_true: Ground truth binary labels
        y_prob: Predicted probabilities
        pct: Percentile (0.01 for 1%, 0.10 for 10%)
        
    Returns:
        Lift value (e.g., 20 means 20x better than random)
    """
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    precision_at_k = y_true[top_k_indices].mean()
    baseline = y_true.mean()
    return precision_at_k / baseline if baseline > 0 else 0.0


def true_positives_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> int:
    """Count true positives in top percentile."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return int(y_true[top_k_indices].sum())


def precision_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """Calculate precision at top percentile (PPV@k)."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return float(y_true[top_k_indices].mean())


def sensitivity_at_percentage(y_true: np.ndarray, y_prob: np.ndarray, pct: float) -> float:
    """
    Calculate sensitivity at top percentile.
    
    Sensitivity = TP / (TP + FN) for members in top k%
    """
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    
    # Binary prediction: 1 for top k%, 0 for rest
    y_pred_binary = np.zeros(n, dtype=int)
    y_pred_binary[top_k_indices] = 1
    
    tp = (y_true[top_k_indices] == 1).sum()
    fn = ((y_true == 1) & (y_pred_binary == 0)).sum()
    
    return float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0


def compute_split_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    """
    Compute all evaluation metrics for a single split.
    
    This function replicates the exact metrics from the original Medicaid IP model:
    - ROC-AUC: Overall discrimination ability
    - AUC-PR: Precision-Recall AUC (important for imbalanced data)
    - Brier Score: Calibration metric
    - Lift@1%, Lift@5%, Lift@10%: Key business metrics
    - PPV@1%, PPV@10%: Precision at top percentiles
    - Sensitivity@1%, Sensitivity@10%: Recall at top percentiles
    - TP@1%: True positives captured in top 1%
    
    Args:
        y_true: Ground truth labels
        y_prob: Predicted probabilities
        
    Returns:
        Dict with metric names as keys
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    
    return {
        # Discrimination metrics
        'auc_roc': roc_auc_score(y_true, y_prob),
        'auc_pr': average_precision_score(y_true, y_prob),
        'brier': brier_score_loss(y_true, y_prob),
        
        # Lift metrics (primary business metrics)
        'lift_1pct': lift_at_percentage(y_true, y_prob, 0.01),
        'lift_5pct': lift_at_percentage(y_true, y_prob, 0.05),
        'lift_10pct': lift_at_percentage(y_true, y_prob, 0.10),
        
        # PPV (Precision) at percentiles
        'ppv_1pct': precision_at_percentage(y_true, y_prob, 0.01) * 100,
        'ppv_10pct': precision_at_percentage(y_true, y_prob, 0.10) * 100,
        
        # Sensitivity (Recall) at percentiles
        'sensitivity_1pct': sensitivity_at_percentage(y_true, y_prob, 0.01) * 100,
        'sensitivity_10pct': sensitivity_at_percentage(y_true, y_prob, 0.10) * 100,
        
        # True positives captured
        'tp_1pct': true_positives_at_percentage(y_true, y_prob, 0.01),
        
        # Sample info
        'n_samples': len(y_true),
        'n_positives': int(y_true.sum()),
        'prevalence': float(y_true.mean()),
    }


#### Load data

In [91]:
# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

def load_medicaid_heldout_data(
    sample_frac: Optional[float] = None,
    random_state: int = RANDOM_STATE,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Load HELDOUT Medicaid IP data from BigQuery for downstream evaluation.
    
    This function loads from the HELDOUT tables which contain members
    NOT in the TE pretraining 10% sample, ensuring no data leakage.
    
    Tables loaded:
    - HELDOUT_FEATURES_TABLE: Non-embedding features (already filtered)
    - HELDOUT_OUTCOME_TABLE: Target variable (acute_ip_flag)
    
    Note: Embeddings are NOT loaded here - they must be generated via
    transformer inference on HELDOUT_TE_INPUT_TABLE and merged separately.
    
    Args:
        sample_frac: Optional sampling fraction for testing (e.g., 0.1 for 10%)
        random_state: Random seed for sampling
        verbose: Print progress information
        
    Returns:
        DataFrame with features and outcome (no embeddings)
    """
    client = bigquery.Client()
    
    if verbose:
        print(f"\n{'='*70}")
        print("LOADING MEDICAID IP HELDOUT DATA FROM BIGQUERY")
        print(f"{'='*70}")
        print(f"Features table: {HELDOUT_FEATURES_TABLE}")
        print(f"Outcome table: {HELDOUT_OUTCOME_TABLE}")
    
    # Step 1: Load heldout non-embedding features (already filtered)
    if verbose:
        print("\n[Step 1/3] Loading heldout non-embedding features...")
    
    features_sql = f"""
    SELECT *
    FROM `{HELDOUT_FEATURES_TABLE}`
    """
    
    df_features = client.query(features_sql).to_dataframe()
    if verbose:
        print(f"  Features loaded: {len(df_features):,} rows, {len(df_features.columns)} columns")
    
    # Step 2: Load heldout outcomes
    if verbose:
        print("\n[Step 2/3] Loading heldout outcomes...")
    
    outcomes_sql = f"""
    SELECT
        asdb_member_key,
        acute_ip_flag
    FROM `{HELDOUT_OUTCOME_TABLE}`
    """
    
    df_outcomes = client.query(outcomes_sql).to_dataframe()
    if verbose:
        print(f"  Outcomes loaded: {len(df_outcomes):,} rows")
        print(f"  Positive rate: {df_outcomes[TARGET_COLUMN].mean()*100:.2f}%")
    
    # Step 3: Merge features with outcomes
    if verbose:
        print("\n[Step 3/3] Merging features and outcomes...")
    
    df_features = df_features.set_index(MEMBER_KEY)
    df_outcomes = df_outcomes.set_index(MEMBER_KEY)
    
    df_merged = df_features.merge(df_outcomes, left_index=True, right_index=True, how='inner')
    df_merged = df_merged.reset_index()
    
    if verbose:
        print(f"  Merged dataset: {len(df_merged):,} rows, {len(df_merged.columns)} columns")
    
    # Optional sampling for testing
    if sample_frac is not None:
        if verbose:
            print(f"\n  Sampling {sample_frac*100:.0f}% of data...")
        df_merged = df_merged.sample(frac=sample_frac, random_state=random_state)
        if verbose:
            print(f"  Sampled dataset: {len(df_merged):,} rows")
    
    if verbose:
        print(f"\n✅ Data loading complete!")
        print(f"   Final shape: {df_merged.shape}")
        print(f"   Positive rate: {df_merged[TARGET_COLUMN].mean()*100:.2f}%")
        print(f"   ⚠️  Note: Embeddings NOT included - merge from NPZ after inference")
        print(f"{'='*70}\n")
    
    return df_merged


def load_te_inference_input(
    sample_frac: Optional[float] = None,
    random_state: int = RANDOM_STATE,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Load TE inference input data for generating embeddings for heldout members.
    
    This table contains the raw TE sequences (cd, gender_cd, age_in_months)
    needed to run transformer inference and generate embeddings.
    
    Returns:
        DataFrame with columns: asdb_member_key, individual_id, index_dt,
        gender_cd, age_in_months, cd, dt_cnt
    """
    client = bigquery.Client()
    
    if verbose:
        print(f"\n{'='*70}")
        print("LOADING TE INFERENCE INPUT FOR HELDOUT MEMBERS")
        print(f"{'='*70}")
        print(f"Table: {HELDOUT_TE_INPUT_TABLE}")
    
    sql = f"""
    SELECT *
    FROM `{HELDOUT_TE_INPUT_TABLE}`
    """
    
    df = client.query(sql).to_dataframe()
    
    if verbose:
        print(f"  Loaded: {len(df):,} rows")
        print(f"  Unique members: {df[MEMBER_KEY].nunique():,}")
        print(f"  Avg sequence days: {df['dt_cnt'].mean():.1f}")
    
    # Optional sampling
    if sample_frac is not None:
        if verbose:
            print(f"\n  Sampling {sample_frac*100:.0f}% of data...")
        df = df.sample(frac=sample_frac, random_state=random_state)
        if verbose:
            print(f"  Sampled: {len(df):,} rows")
    
    if verbose:
        print(f"{'='*70}\n")
    
    return df


# Legacy function for backward compatibility with original Eric Ma pipeline
def load_medicaid_data_from_bigquery(
    sample_frac: Optional[float] = None,
    random_state: int = RANDOM_STATE,
    verbose: bool = True
) -> pd.DataFrame:
    """
    DEPRECATED: Use load_medicaid_heldout_data() for downstream evaluation.
    
    This function is kept for backward compatibility but now loads from
    heldout tables. For the full pipeline, use load_medicaid_heldout_data()
    and merge embeddings separately.
    """
    if verbose:
        print("⚠️  Note: load_medicaid_data_from_bigquery() now loads from HELDOUT tables")
        print("   For new embeddings, use load_medicaid_heldout_data() + merge_new_embeddings_with_features()")
    
    return load_medicaid_heldout_data(
        sample_frac=sample_frac,
        random_state=random_state,
        verbose=verbose
    )




#### Data and feature preprocessing

In [92]:
def preprocess_features(
    df: pd.DataFrame,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Preprocess features replicating the original Medicaid IP pipeline.
    
    Preprocessing steps:
    1. Fill embedding columns (embedding_0-embedding_255) with 0
    2. Fill numeric columns with 0
    3. Fill string/categorical columns with empty string
    4. Encode gender: M -> 1, F -> 0, other -> -1
    
    Note: CatBoost handles categorical features natively, so we don't
    need to one-hot encode them. We just ensure proper types.
    
    Args:
        df: Raw DataFrame
        verbose: Print progress
        
    Returns:
        Preprocessed DataFrame
    """
    import re
    from pandas.api.types import is_integer_dtype, is_float_dtype
    
    df = df.copy()
    
    if verbose:
        print("Preprocessing features...")
    
    # Step 1: Fill embedding columns with 0
    emb_pattern = r'^embedding_\d+$'
    emb_cols = [col for col in df.columns if re.match(emb_pattern, col)]
    if emb_cols:
        df[emb_cols] = df[emb_cols].fillna(0)
        if verbose:
            print(f"  Filled {len(emb_cols)} embedding columns with 0")
    
    # Step 2: Fill numeric columns with 0, string columns with ''
    numeric_filled = 0
    string_filled = 0
    
    for col in df.columns:
        if col in emb_cols or col == TARGET_COLUMN or col == MEMBER_KEY:
            continue
            
        if is_integer_dtype(df[col]) or is_float_dtype(df[col]):
            df[col] = df[col].fillna(0)
            numeric_filled += 1
        else:
            try:
                df[col] = df[col].fillna('')
                string_filled += 1
            except Exception as e:
                if verbose:
                    print(f"  Warning: Could not process column {col}: {e}")
    
    if verbose:
        print(f"  Filled {numeric_filled} numeric columns with 0")
        print(f"  Filled {string_filled} string columns with ''")
    
    # Step 3: Encode gender (matches original)
    if 'gender' in df.columns:
        df['gender'] = df['gender'].map({'M': 1, 'F': 0}).fillna(-1).astype(int)
        if verbose:
            print("  Encoded gender: M->1, F->0, other->-1")
    
    if verbose:
        print("✅ Preprocessing complete!")
    
    return df


def downsample_negatives(
    X: pd.DataFrame,
    y: pd.Series,
    ratio: float = CATBOOST_UNDERSAMPLE_RATIO,
    random_state: int = UNDERSAMPLE_RANDOM_STATE  # Eric uses 53 for undersampling
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Downsample negative class to achieve target ratio.
    
    The original Medicaid IP model used undersampling as the primary
    class imbalance strategy. CatBoost worked best with 0.2 ratio
    (20% minority, meaning 5:1 negative-to-positive ratio).
    
    Args:
        X: Feature DataFrame
        y: Target Series
        ratio: Minority class ratio (e.g., 0.2 for 5:1)
        random_state: Random seed
        
    Returns:
        Tuple of (X_resampled, y_resampled)
    """
    np.random.seed(random_state)
    
    pos_mask = y == 1
    neg_mask = y == 0
    
    pos_indices = X.index[pos_mask].tolist()
    neg_indices = X.index[neg_mask].tolist()
    
    n_positives = len(pos_indices)
    n_negatives = len(neg_indices)
    
    # Calculate target number of negatives based on ratio
    # ratio = n_positives / (n_positives + n_negatives_target)
    # Solving: n_negatives_target = n_positives * (1 - ratio) / ratio
    target_n_negatives = int(n_positives * (1 - ratio) / ratio)
    
    if n_negatives <= target_n_negatives:
        print(f"  Downsampling: No action needed (current ratio: {n_positives/(n_positives+n_negatives):.3f})")
        return X, y
    
    # Randomly sample negatives
    sampled_neg_indices = np.random.choice(neg_indices, size=target_n_negatives, replace=False)
    keep_indices = pos_indices + sampled_neg_indices.tolist()
    
    X_resampled = X.loc[keep_indices].copy()
    y_resampled = y.loc[keep_indices].copy()
    
    # Shuffle
    shuffle_idx = np.random.permutation(len(X_resampled))
    X_resampled = X_resampled.iloc[shuffle_idx].reset_index(drop=True)
    y_resampled = y_resampled.iloc[shuffle_idx].reset_index(drop=True)
    
    new_ratio = y_resampled.sum() / len(y_resampled)
    print(f"  Downsampling: {n_negatives}:{n_positives} -> "
          f"{target_n_negatives}:{n_positives} (ratio: {new_ratio:.3f})")
    
    return X_resampled, y_resampled


# =============================================================================
# DATA PREPARATION
# =============================================================================

@dataclass
class MedicaidPreparedData:
    """
    Container for prepared Medicaid IP evaluation data.
    
    Prepare data once using prepare_medicaid_evaluation_data(), 
    then evaluate multiple models using evaluate_with_prepared_data().
    """
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_val: pd.Series
    y_test: pd.Series
    feature_cols: List[str]
    embedding_features: List[str]
    tabular_features: List[str]
    cat_feature_indices: List[int]
    feature_set: str
    downsampled: bool
    original_train_size: int
    

def create_train_val_test_split(
    df: pd.DataFrame,
    test_size: float = TEST_SIZE,
    val_size: float = VAL_SIZE,
    random_state: int = RANDOM_STATE,
    verbose: bool = True
) -> Dict[str, pd.DataFrame]:
    """
    Create stratified train/validation/test splits.
    
    Replicates the original Medicaid IP model split strategy:
    - 80% train, 10% validation, 10% test
    - Stratified by target variable to preserve class distribution
    
    Args:
        df: Full DataFrame with features and target
        test_size: Fraction for test set (default 0.1)
        val_size: Fraction for validation set (default 0.1)
        random_state: Random seed
        verbose: Print info
        
    Returns:
        Dict with 'train', 'val', 'test' DataFrames
    """
    if verbose:
        print("\nCreating stratified train/val/test splits...")
    
    # First split: train+val vs test
    train_val, test = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=df[TARGET_COLUMN]
    )
    
    # Second split: train vs val
    val_size_adjusted = val_size / (1 - test_size)  # Adjust for remaining data
    train, val = train_test_split(
        train_val,
        test_size=val_size_adjusted,
        random_state=random_state,
        stratify=train_val[TARGET_COLUMN]
    )
    
    splits = {
        'train': train.reset_index(drop=True),
        'val': val.reset_index(drop=True),
        'test': test.reset_index(drop=True),
    }
    
    if verbose:
        for name, split_df in splits.items():
            prevalence = split_df[TARGET_COLUMN].mean() * 100
            print(f"  {name}: {len(split_df):,} rows, "
                  f"{int(split_df[TARGET_COLUMN].sum()):,} positives ({prevalence:.2f}%)")
    
    return splits


def prepare_medicaid_evaluation_data(
    df: pd.DataFrame,
    feature_set: str = 'hybrid',
    apply_downsampling: bool = True,
    downsample_ratio: float = CATBOOST_UNDERSAMPLE_RATIO,
    split_random_state: int = RANDOM_STATE,
    undersample_random_state: int = UNDERSAMPLE_RANDOM_STATE,
    verbose: bool = True
) -> MedicaidPreparedData:
    """
    Prepare Medicaid IP data for model evaluation.
    
    This function encapsulates the complete data preparation pipeline
    from the original Medicaid IP model:
    1. Preprocessing (missing values, encoding)
    2. Train/val/test splitting (stratified)
    3. Feature selection based on feature_set
    4. Optional downsampling of training set
    
    Args:
        df: Raw Medicaid IP DataFrame
        feature_set: One of 'embedding_only', 'tabular_only', 'hybrid'
        apply_downsampling: Whether to downsample training set
        downsample_ratio: Undersampling ratio (default 0.2 for CatBoost)
        split_random_state: Random seed for train/test split (default 35, Eric's)
        undersample_random_state: Random seed for undersampling (default 53, Eric's)
        verbose: Print progress
        
    Returns:
        MedicaidPreparedData object ready for model evaluation
    """
    start_time = time.time()
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"PREPARING MEDICAID IP DATA FOR EVALUATION")
        print(f"{'='*70}")
        print(f"Feature set: {feature_set}")
        print(f"Downsampling: {apply_downsampling} (ratio: {downsample_ratio})")
    
    # Validate feature_set
    valid_feature_sets = {'embedding_only', 'tabular_only', 'hybrid'}
    if feature_set not in valid_feature_sets:
        raise ValueError(f"feature_set must be one of {valid_feature_sets}")
    
    # Step 1: Preprocess
    if verbose:
        print("\n[Step 1/4] Preprocessing features...")
    df_processed = preprocess_features(df, verbose=verbose)
    
    # Step 2: Split data (using split_random_state=35 to match Eric's train_test_split)
    if verbose:
        print("\n[Step 2/4] Creating train/val/test splits...")
    splits = create_train_val_test_split(df_processed, random_state=split_random_state, verbose=verbose)
    
    # Step 3: Select features based on feature_set
    if verbose:
        print(f"\n[Step 3/4] Selecting features for '{feature_set}'...")
    
    # Identify available features
    available_tabular = [f for f in SELECTED_TABULAR_FEATURES if f in df_processed.columns]
    available_embedding = [f for f in EMBEDDING_FEATURES if f in df_processed.columns]
    
    if verbose:
        print(f"  Available tabular features: {len(available_tabular)}")
        print(f"  Available embedding features: {len(available_embedding)}")
    
    if feature_set == 'embedding_only':
        feature_cols = available_embedding
    elif feature_set == 'tabular_only':
        feature_cols = available_tabular
    else:  # hybrid
        feature_cols = available_tabular + available_embedding
    
    if verbose:
        print(f"  Selected features: {len(feature_cols)}")
    
    # Prepare X and y for each split
    X_train = splits['train'][feature_cols].copy()
    X_val = splits['val'][feature_cols].copy()
    X_test = splits['test'][feature_cols].copy()
    y_train = splits['train'][TARGET_COLUMN].astype(int)
    y_val = splits['val'][TARGET_COLUMN].astype(int)
    y_test = splits['test'][TARGET_COLUMN].astype(int)
    
    original_train_size = len(X_train)
    
    # Step 4: Apply downsampling to training set (using undersample_random_state=53 to match Eric's)
    downsampled = False
    if apply_downsampling:
        if verbose:
            print(f"\n[Step 4/4] Applying downsampling to training set...")
        X_train, y_train = downsample_negatives(
            X_train, y_train, 
            ratio=downsample_ratio, 
            random_state=undersample_random_state  # Eric uses 53 for undersampling
        )
        downsampled = True
    else:
        if verbose:
            print(f"\n[Step 4/4] Skipping downsampling...")
    
    # Identify categorical columns for CatBoost
    cat_feature_indices = []
    if feature_set != 'embedding_only':
        cat_cols = [c for c in CATEGORICAL_FEATURES if c in feature_cols]
        cat_feature_indices = [feature_cols.index(c) for c in cat_cols if c in feature_cols]
        if verbose:
            print(f"  Categorical features for CatBoost: {len(cat_feature_indices)}")
    
    elapsed = time.time() - start_time
    
    if verbose:
        print(f"\n✅ Data preparation complete! ({elapsed:.1f}s)")
        print(f"   Train: {len(X_train):,} samples ({y_train.sum():,} positives)")
        print(f"   Val: {len(X_val):,} samples ({y_val.sum():,} positives)")
        print(f"   Test: {len(X_test):,} samples ({y_test.sum():,} positives)")
        print(f"{'='*70}\n")
    
    return MedicaidPreparedData(
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        feature_cols=feature_cols,
        embedding_features=available_embedding,
        tabular_features=available_tabular,
        cat_feature_indices=cat_feature_indices,
        feature_set=feature_set,
        downsampled=downsampled,
        original_train_size=original_train_size,
    )


# =============================================================================
# MODEL EVALUATION
# =============================================================================

def evaluate_model_on_splits(
    model: Any,
    prepared_data: MedicaidPreparedData,
    verbose: bool = True
) -> Dict[str, Dict[str, float]]:
    """
    Train model on training set and evaluate on all splits.
    
    Args:
        model: CatBoostClassifier or compatible model
        prepared_data: MedicaidPreparedData from prepare_medicaid_evaluation_data()
        verbose: Print progress
        
    Returns:
        Dict with split names as keys, metrics dict as values
    """
    start_time = time.time()
    
    # Clone model to avoid modifying original
    model = clone(model)
    model_type = type(model).__name__
    
    if verbose:
        print(f"\nTraining {model_type}...")
    
    # Train model
    if model_type == 'CatBoostClassifier':
        train_pool = Pool(
            prepared_data.X_train, 
            prepared_data.y_train,
            cat_features=prepared_data.cat_feature_indices if prepared_data.cat_feature_indices else None
        )
        val_pool = Pool(
            prepared_data.X_val,
            prepared_data.y_val,
            cat_features=prepared_data.cat_feature_indices if prepared_data.cat_feature_indices else None
        )
        model.fit(train_pool, eval_set=val_pool, verbose=0)
    else:
        model.fit(prepared_data.X_train, prepared_data.y_train)
    
    train_time = time.time() - start_time
    if verbose:
        print(f"  Training completed in {train_time:.1f}s")
    
    # Evaluate on all splits
    results = {}
    splits_data = {
        'train': (prepared_data.X_train, prepared_data.y_train),
        'val': (prepared_data.X_val, prepared_data.y_val),
        'test': (prepared_data.X_test, prepared_data.y_test),
    }
    
    for split_name, (X_split, y_split) in splits_data.items():
        if verbose:
            print(f"  Evaluating on {split_name}...")
        
        # Predict probabilities
        if model_type == 'CatBoostClassifier' and prepared_data.cat_feature_indices:
            pool = Pool(X_split, cat_features=prepared_data.cat_feature_indices)
            y_prob = model.predict_proba(pool)[:, 1]
        else:
            y_prob = model.predict_proba(X_split)[:, 1]
        
        # Compute metrics
        results[split_name] = compute_split_metrics(np.array(y_split), y_prob)
    
    # Add training time
    results['_training_time_sec'] = train_time
    
    return results


def evaluate_with_prepared_data(
    prepared_data: MedicaidPreparedData,
    model: Any,
    exp_name: str,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Evaluate a model using pre-prepared data.
    
    Args:
        prepared_data: MedicaidPreparedData from prepare_medicaid_evaluation_data()
        model: Pre-configured model (e.g., CatBoostClassifier)
        exp_name: Experiment name for result identification
        verbose: Print progress
        
    Returns:
        Dict with exp_name, model_type, feature_set, and all metrics
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"EVALUATING: {exp_name}")
        print(f"{'='*70}")
    
    # Evaluate model
    split_results = evaluate_model_on_splits(model, prepared_data, verbose=verbose)
    
    # Build output dictionary
    output = {
        'exp_name': exp_name,
        'model_type': type(model).__name__,
        'feature_set': prepared_data.feature_set,
        'n_features': len(prepared_data.feature_cols),
        'n_embedding_features': len(prepared_data.embedding_features),
        'n_tabular_features': len(prepared_data.tabular_features),
        'downsampled': prepared_data.downsampled,
        'original_train_size': prepared_data.original_train_size,
        'actual_train_size': len(prepared_data.X_train),
        'training_time_sec': split_results.pop('_training_time_sec', 0),
    }
    
    # Flatten split results with prefixes
    for split_name, metrics in split_results.items():
        for metric_name, value in metrics.items():
            output[f'{split_name}_{metric_name}'] = value
    
    if verbose:
        print(f"\n📊 Key Results for {exp_name}:")
        print(f"   Test AUC-ROC: {output.get('test_auc_roc', 0):.4f}")
        print(f"   Test Lift@1%: {output.get('test_lift_1pct', 0):.2f}x")
        print(f"   Test Lift@10%: {output.get('test_lift_10pct', 0):.2f}x")
        print(f"   Test PPV@1%: {output.get('test_ppv_1pct', 0):.2f}%")
        print(f"{'='*70}\n")
    
    return output


# =============================================================================
# EMBEDDING INTEGRATION
# =============================================================================

def load_embeddings_from_npz(
    embedding_path: str
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load embeddings from NPZ file (generated by embedding generation pipeline).
    
    Args:
        embedding_path: Path to NPZ file or directory containing NPZ files
        
    Returns:
        Tuple of (embeddings, individual_ids, index_dts)
    """
    if os.path.isdir(embedding_path):
        npz_files = glob.glob(os.path.join(embedding_path, "embeddings_*.npz"))
        if not npz_files:
            raise FileNotFoundError(f"No NPZ files found in {embedding_path}")
        npz_path = sorted(npz_files)[-1]  # Use most recent
    else:
        npz_path = embedding_path
    
    print(f"Loading embeddings from: {npz_path}")
    data = np.load(npz_path, allow_pickle=True)
    
    return (
        data['embeddings'],
        data['individual_ids'],
        data['index_dts']
    )


def load_embeddings_from_bigquery(
    embedding_table: str,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Load embeddings from BigQuery table.
    
    Args:
        embedding_table: Full BigQuery table path
        verbose: Print progress
        
    Returns:
        DataFrame with asdb_member_key, index_dt, and emb0-emb255 columns
    """
    client = bigquery.Client()
    
    if verbose:
        print(f"Loading embeddings from: {embedding_table}")
    
    sql = f"""
    SELECT *
    FROM `{embedding_table}`
    """
    
    df_embeddings = client.query(sql).to_dataframe()
    
    if verbose:
        print(f"  Loaded: {len(df_embeddings):,} rows, {len(df_embeddings.columns)} columns")
        emb_cols = [c for c in df_embeddings.columns if c.startswith('emb')]
        print(f"  Embedding dimensions: {len(emb_cols)}")
    
    return df_embeddings


def merge_embeddings_with_features_outcomes(
    df_features: pd.DataFrame,
    df_outcome: pd.DataFrame,
    df_embeddings: pd.DataFrame,
    features_key='asdb_member_key',    # Medicaid native key
    embeddings_key='individual_id',     # TE pipeline key
    verbose: bool = True
) -> pd.DataFrame:
    """
    Merge embeddings with features and outcomes for downstream evaluation.
    
    Args:
        df_features: Features DataFrame (from heldout features table)
        df_outcome: Outcomes DataFrame (from heldout outcome table)
        df_embeddings: Embeddings DataFrame (from BigQuery embedding table)
        merge_key: Column to join on (default: asdb_member_key)
        verbose: Print progress
        
    Returns:
        Merged DataFrame ready for evaluation
    """
    if verbose:
        print(f"\nMerging data...")
        print(f"  Features: {len(df_features):,} rows (key: {features_key})")
        print(f"  Outcomes: {len(df_outcome):,} rows (key: {features_key})")
        print(f"  Embeddings: {len(df_embeddings):,} rows (key: {embeddings_key})")
    
    # Ensure key types match (as strings)
    df_features[features_key] = df_features[features_key].astype(str)
    df_outcome[features_key] = df_outcome[features_key].astype(str)
    df_embeddings[embeddings_key] = df_embeddings[embeddings_key].astype(str)
    
    # Merge features with outcomes (same key)
    df_merged = df_features.merge(
        df_outcome[[features_key, TARGET_COLUMN]], 
        on=features_key, 
        how='inner'
    )
    
    if verbose:
        print(f"  After features+outcomes merge: {len(df_merged):,} rows")
    
    # Extract embedding columns
    emb_cols = [c for c in df_embeddings.columns if c.startswith('emb')]
    
    # Merge with embeddings using different keys
    # Rename embeddings key to match features key for the merge
    df_emb_subset = df_embeddings[[embeddings_key] + emb_cols].copy()
    df_emb_subset = df_emb_subset.rename(columns={embeddings_key: features_key})
    
    df_merged = df_merged.merge(df_emb_subset, on=features_key, how='inner')
    
    if verbose:
        print(f"  After embeddings merge: {len(df_merged):,} rows")
        print(f"  Total columns: {len(df_merged.columns)}")
        print(f"  Positive rate: {df_merged[TARGET_COLUMN].mean()*100:.2f}%")
    
    return df_merged


#### Generic pipeline

In [82]:

# =============================================================================
# HIGH-LEVEL EVALUATION FUNCTIONS
# =============================================================================

def run_medicaid_ip_evaluation(
    embedding_path: Optional[str] = None,
    feature_set: str = 'hybrid',
    sample_frac: Optional[float] = None,
    use_tuned_params: bool = True,
    apply_downsampling: bool = True,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Run complete Medicaid IP evaluation pipeline.
    
    This is the main entry point for evaluating embeddings on the
    Medicaid IP hospitalization prediction task.
    
    Args:
        embedding_path: Path to new embeddings NPZ file (None = use original)
        feature_set: 'embedding_only', 'tabular_only', or 'hybrid'
        sample_frac: Sample fraction for testing (None = full data)
        use_tuned_params: Use Optuna-tuned hyperparameters
        apply_downsampling: Apply class imbalance handling
        verbose: Print progress
        
    Returns:
        Dict with all evaluation results
    """
    start_time = time.time()
    
    if verbose:
        print(f"\n{'#'*70}")
        print("MEDICAID IP DOWNSTREAM EVALUATION")
        print(f"{'#'*70}")
        print(f"Feature set: {feature_set}")
        print(f"New embeddings: {embedding_path or 'Using original'}")
        print(f"Sample fraction: {sample_frac or 'Full data'}")
    
    # Step 1: Load data from BigQuery
    df = load_medicaid_data_from_bigquery(
        sample_frac=sample_frac,
        verbose=verbose
    )
    
    # Step 2: Replace embeddings if new path provided
    if embedding_path is not None:
        if verbose:
            print("\nReplacing embeddings with new transformer embeddings...")
        embeddings, individual_ids, index_dts = load_embeddings_from_npz(embedding_path)
        df = merge_new_embeddings_with_features(df, embeddings, individual_ids)
    
    # Step 3: Prepare data
    prepared_data = prepare_medicaid_evaluation_data(
        df=df,
        feature_set=feature_set,
        apply_downsampling=apply_downsampling,
        verbose=verbose
    )
    
    # Step 4: Configure model
    if use_tuned_params:
        model = CatBoostClassifier(**CATBOOST_TUNED_PARAMS)
    else:
        model = CatBoostClassifier(**CATBOOST_BALANCED_PARAMS)
    
    # Step 5: Evaluate
    exp_name = f"medicaid_ip_{feature_set}"
    if embedding_path:
        exp_name += f"_new_emb"
    
    results = evaluate_with_prepared_data(
        prepared_data=prepared_data,
        model=model,
        exp_name=exp_name,
        verbose=verbose
    )
    
    # Add metadata
    results['embedding_path'] = embedding_path
    results['sample_frac'] = sample_frac
    results['use_tuned_params'] = use_tuned_params
    results['total_time_sec'] = time.time() - start_time
    
    return results


#### Adapted embedding saving

In [84]:
def save_medicaid_embeddings(
    embeddings: np.ndarray,
    member_keys: List[str],
    index_dts: List[str],
    output_path: str,
    model_name: str = "",
    additional_metadata: Dict = None
) -> str:
    """
    Save Medicaid embeddings to disk in NPZ format.
    Thin wrapper around save_embeddings with Medicaid-specific metadata.
    """
    # Add Medicaid-specific metadata
    metadata = {
        'lob': 'Medicaid',
        'member_key_type': 'asdb_member_key',
        **(additional_metadata or {})
    }
    
    # Use the unified save function
    return save_embeddings(
        embeddings=embeddings,
        individual_ids=member_keys,  # Named for compatibility, but contains asdb_member_key
        index_dts=index_dts,
        output_path=output_path,
        model_name=f"medicaid_{model_name}" if model_name else "medicaid",
        additional_metadata=metadata
    )


def save_medicaid_embeddings_to_bigquery(
    embeddings: np.ndarray,
    member_keys: List[str],
    index_dts: List[str],
    project_id: str = PROJECT_ID,
    dataset_id: str = DATASET_ID,
    table_name: str = "",
    exp_name: str = "",
    model_type: str = "",
    if_exists: str = "replace"
) -> str:
    """
    Save Medicaid embeddings to BigQuery.
    Thin wrapper with Medicaid-specific column naming.
    """
    return save_embeddings_to_bigquery(
        embeddings=embeddings,
        individual_ids=member_keys,  # Named for compatibility
        index_dts=index_dts,
        project_id=project_id,
        dataset_id=dataset_id,
        table_name=table_name,
        exp_name=exp_name,
        model_type=model_type,
        if_exists=if_exists
    )

#### Embedding generation

In [36]:
medicaid_sql = f"""
SELECT *
FROM `{HELDOUT_TE_INPUT_TABLE}`
"""
client = bigquery.Client()
df_te_input = client.query(medicaid_sql).to_dataframe()

In [39]:
# Get OOT dataset
df_te_input['index_dt'] = pd.to_datetime(df_te_input['index_dt'])
df_pre_oot = df_te_input[df_te_input['index_dt'] <= pd.to_datetime(OOT_CUTOFF_DATE)]
df_oot = df_te_input[df_te_input['index_dt'] > pd.to_datetime(OOT_CUTOFF_DATE)]
print(f"\nTime distribution (OOT cutoff: {OOT_CUTOFF_DATE}):")
print(f"  Pre-OOT (≤{OOT_CUTOFF_DATE}): {len(df_pre_oot):,} members")
print(f"  OOT (>{OOT_CUTOFF_DATE}): {len(df_oot):,} members")

In [60]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = {} 
cleanup_gpu_memory(verbose=True)
for exp_name, model_path in tqdm(MODEL_PATHS.items(), desc="Processing models"):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp_name}")
    print(f"{'='*70}")

    cleanup_gpu_memory(verbose=False)

    # Load model
    model, config, moe_config, use_mixed_precision, model_type = load_model_from_checkpoint(
        model_path=model_path,
        device=device
    )

    # Generate embeddings
    inference_start = time.time()
    embeddings, member_keys, index_dts = generate_embeddings(
        model=model,
        config=config,
        data=df_te_input,
        device=device,
        # ========== MEDICAID-SPECIFIC PARAMETERS ==========
        id_column='asdb_member_key',   # Medicaid's primary key
        lob_value='Medicaid',           # Add lob column if missing
        desc_prefix='Medicaid',         # Progress bar prefix
        # ==================================================
        batch_size=64,
        num_workers=4,
        use_mixed_precision=use_mixed_precision,
        verbose=True,
        multi_gpu=True,
        moe_config=moe_config,
    )
    inference_duration = time.time() - inference_start

    safe_exp_name = exp_name.replace('-', '_').replace('.', '_')
    table_name = f"a964286_te4exp_{safe_exp_name}_medicaid_heldout_embedding"
    bq_table_path = None
    bq_table_path = save_medicaid_embeddings_to_bigquery(
        embeddings=embeddings,
        member_keys=member_keys,
        index_dts=index_dts,
        table_name=table_name,
        exp_name=exp_name,
        model_type=model_type,
    )
    results[exp_name] = {
        'bq_table_path': bq_table_path,
        'embedding_shape': embeddings.shape,
        'model_type': model_type,
        'model_path': model_path,
        'inference_duration_sec': inference_duration,
        'inference_duration_hr': round(inference_duration / 3600, 2),
        'status': 'success'
    }
    del model
    del embeddings
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    for exp_name, result in results.items():
        print(f"  {exp_name}: {result['embedding_shape']} ({result['inference_duration_hr']:.2f}hr)")


GPU MEMORY STATUS (4 GPUs)
GPU 0:
  Allocated:     0.21 GB
  Reserved:      0.26 GB
  Peak Allocated: 0.21 GB
  Fragmentation: 19.9%

GPU 1:
  Allocated:     0.11 GB
  Reserved:      0.11 GB
  Peak Allocated: 0.11 GB
  Fragmentation: 4.9%

GPU 2:
  Allocated:     0.11 GB
  Reserved:      0.11 GB
  Peak Allocated: 0.11 GB
  Fragmentation: 4.9%

GPU 3:
  Allocated:     0.11 GB
  Reserved:      0.11 GB
  Peak Allocated: 0.11 GB
  Fragmentation: 4.9%




Processing models:   0%|          | 0/4 [00:00<?, ?it/s]


EXPERIMENT: exp1_dense_baseline_pure_legacy

Loading model from: logs/exp_round5_3lobs_1-5M_pretrain_multi_gpu_test_v2/exp1_dense_baseline_pure_legacy/saved_models/exp_round5_3lobs_pretrain_multi_gpu_test_v2_exp1_dense_baseline_bs128_ep1_d256_20251230_055716_final.pt
  Model type: BaselineTransformer
  Embedding size: 256
  N layers: 6
  Use learned attention pooling: False
✅ Model loaded successfully!
   Total parameters: 26,455,961
   Mixed precision: False
   Device: cuda


MEDICAID EMBEDDING GENERATION
Samples: 2,194,312 | Batch: 64 | GPUs: 4
Workers: 4 | Mixed precision: False
ID column: asdb_member_key
  Added 'lob'='Medicaid' column
Multi-GPU mode: 4 GPUs
  Cloning model to GPU 0...
  Cloning model to GPU 1...
  Cloning model to GPU 2...
  Cloning model to GPU 3...
  GPU 0: samples 0 to 548,578 (548,578 samples)
  GPU 1: samples 548,578 to 1,097,156 (548,578 samples)
  GPU 2: samples 1,097,156 to 1,645,734 (548,578 samples)
  GPU 3: samples 1,645,734 to 2,194,312 (548,578 sampl


Generating Medicaid embeddings (4 GPUs):   0%|          | 0/2194312 [00:00<?, ?it/s]


LazyClinicalDataset initialized with 548,578 samples (lazy loading)


...

LazyClinicalDataset initialized with 548,578 samples (lazy loading)



Generating Medicaid embeddings (4 GPUs): 100%|██████████| 2194312/2194312 [1:56:01<00:00, 315.22it/s][A



✅ Complete! Time: 6964.8s | Speed: 315 samples/s
   Effective: 1,260 samples/s (across 4 GPUs)
   Output: (2194312, 256)
Writing 2,194,312 rows to BigQuery: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_exp6_auxiliary_free_v3_medicaid_heldout_embedding
  Columns: 260 (embedding_dim=256)


Processing models: 100%|██████████| 4/4 [8:27:42<00:00, 7615.61s/it]  

✅ Loaded 2,194,312 rows to edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_exp6_auxiliary_free_v3_medicaid_heldout_embedding
  exp1_dense_baseline_pure_legacy: (2194312, 256) (2.18hr)
  exp1_dense_baseline_opt_config: (2194312, 256) (2.27hr)
  exp2b_flash_learned_pool_v2: (2194312, 256) (1.80hr)
  exp6_auxiliary_free_v3: (2194312, 256) (1.93hr)


In [ ]:
# =============================================================================
# MEDICAID: Embedding Generation for Round 10, 9, and 7 Models
# =============================================================================

MODEL_PATHS_NEW_MEDICAID = {
    'exp_round10_formal_exp2b_flash_learned_pool_d256':
        'logs/exp_round10_3lobs_formal_training/exp2b_flash_learned_pool/saved_models/'
        'exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt',

    'exp_round7_exp2b_flash_learned_pool_d512':
        'logs/exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim/'
        'exp2b_flash_learned_poolexp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final/'
        'saved_models/'
        'exp_round7_3lobs_1-5M_pretrain_multi_gpu_test_v3_512dim_exp2b_flash_learned_pool_bs128_ep1_d512_20260303_023717_final.pt',

    'exp_round9_decoupled_exp2b_flash_learned_pool_v2_d256':
        'logs/exp_round9_3lobs_1-5M_decoupled_training_embedding_v4_256dim/exp2b_flash_learned_pool_v2/saved_models/'
        'exp_round9_3lobs_1-5M_decoupled_training_embedding_v4_256dim_exp2b_flash_learned_pool_bs128_ep1_d256_20260310_123547_final.pt',
}

results_new_medicaid = {}
batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for exp_name, model_path in tqdm(MODEL_PATHS_NEW_MEDICAID.items(), desc="Medicaid embedding generation"):
    cleanup_gpu_memory(verbose=False)
    model, config, moe_config, use_mixed_precision, model_type = load_model_from_checkpoint(
        model_path=model_path,
        device=device,
        verbose=True
    )

    inference_start_time = time.time()
    embeddings, member_keys, index_dts = generate_embeddings(
        model=model,
        config=config,
        data=df_te_input,
        device=device,
        id_column='asdb_member_key',
        lob_value='Medicaid',
        desc_prefix='Medicaid',
        batch_size=batch_size,
        use_mixed_precision=use_mixed_precision,
        verbose=True,
        multi_gpu=True,
        moe_config=moe_config,
    )
    inference_duration = time.time() - inference_start_time
    print(f"Inference duration for {exp_name}: {round(inference_duration/3600, 2):.2f} hr")

    safe_exp_name = exp_name.replace('-', '_').replace('.', '_')
    table_name = f"a964286_te4exp_{safe_exp_name}_medicaid_heldout_embedding"
    bq_table_path = save_medicaid_embeddings_to_bigquery(
        embeddings=embeddings,
        member_keys=member_keys,
        index_dts=index_dts,
        table_name=table_name,
        exp_name=exp_name,
        model_type=model_type,
    )
    results_new_medicaid[exp_name] = {
        'bq_table_path': bq_table_path,
        'embedding_shape': embeddings.shape,
        'model_type': model_type,
        'model_path': model_path,
        'inference_duration_hr': round(inference_duration / 3600, 2),
        'status': 'success'
    }

    del model
    del embeddings
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n=== Medicaid Embedding Generation Summary ===")
for exp_name, result in results_new_medicaid.items():
    print(f"  {exp_name}: {result['embedding_shape']} -> {result['bq_table_path']}")

#### Model training

In [85]:
MEDICAID_EMBEDDING_TABLES = {
    'exp1_dense_baseline_pure_legacy': 
        f"{PROJECT_ID}.{DATASET_ID}.a964286_te4exp_exp1_dense_baseline_pure_legacy_medicaid_heldout_embedding",
    
    'exp1_dense_baseline_opt_config': 
        f"{PROJECT_ID}.{DATASET_ID}.a964286_te4exp_exp1_dense_baseline_opt_config_medicaid_heldout_embedding",
    
    'exp2b_flash_learned_pool_v2': 
        f"{PROJECT_ID}.{DATASET_ID}.a964286_te4exp_exp2b_flash_learned_pool_v2_medicaid_heldout_embedding",
    
    'exp6_auxiliary_free_v3': 
        f"{PROJECT_ID}.{DATASET_ID}.a964286_te4exp_exp6_auxiliary_free_v3_medicaid_heldout_embedding",
}


In [63]:
# =============================================================================
# STEP 1: Load heldout features and outcomes (same for all experiments)
# =============================================================================
client = bigquery.Client()

# Load features
features_sql = f"SELECT * FROM `{HELDOUT_FEATURES_TABLE}`"
df_features = client.query(features_sql).to_dataframe()
print(f"  Features loaded: {len(df_features):,} rows, {len(df_features.columns)} columns")

# Load outcomes
outcomes_sql = f"SELECT {MEMBER_KEY}, {TARGET_COLUMN} FROM `{HELDOUT_OUTCOME_TABLE}`"
df_outcome = client.query(outcomes_sql).to_dataframe()
print(f"  Outcomes loaded: {len(df_outcome):,} rows")
print(f"  Positive rate: {df_outcome[TARGET_COLUMN].mean()*100:.2f}%")

  Features loaded: 3,349,194 rows, 310 columns
  Outcomes loaded: 3,349,194 rows
  Positive rate: 1.68%


In [66]:
df_embeddings = load_embeddings_from_bigquery(f"{PROJECT_ID}.{DATASET_ID}.a964286_te4exp_exp1_dense_baseline_pure_legacy_medicaid_heldout_embedding", verbose=True)

Loading embeddings from: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_exp1_dense_baseline_pure_legacy_medicaid_heldout_embedding
  Loaded: 2,194,312 rows, 260 columns
  Embedding dimensions: 256


In [68]:
df_embeddings.head()

,individual_id,index_dt,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,...,embedding_248,embedding_249,embedding_250,embedding_251,embedding_252,embedding_253,embedding_254,embedding_255,exp_name,model_type
0,461025740,2023-01-01,0.221058,0.457981,0.266327,-1.177322,1.199747,0.208784,0.625673,0.755485,...,-0.943469,-0.036214,-0.548549,-0.136619,0.777296,-0.016145,-0.151912,-2.300765,exp1_dense_baseline_pure_legacy,BaselineTransformer
1,520136419,2023-01-01,0.199575,0.451851,0.281016,-1.146861,1.182900,0.198194,0.611587,0.763375,...,-0.964127,-0.039115,-0.566514,-0.122105,0.767196,-0.054735,-0.168844,-2.292844,exp1_dense_baseline_pure_legacy,BaselineTransformer
2,510365239,2023-01-01,0.207613,0.470398,0.250559,-1.135723,1.195928,0.189500,0.624074,0.764288,...,-0.961297,-0.033395,-0.543723,-0.111997,0.792612,-0.048569,-0.142794,-2.269546,exp1_dense_baseline_pure_legacy,BaselineTransformer
3,460877834,2023-01-01,0.219167,0.454034,0.275286,-1.144062,1.187145,0.207763,0.609709,0.769249,...,-0.941078,-0.034841,-0.553301,-0.130928,0.774184,-0.047355,-0.149109,-2.317828,exp1_dense_baseline_pure_legacy,BaselineTransformer
4,351666089,2023-01-01,0.215571,0.443582,0.258199,-1.171936,1.187262,0.183399,0.623998,0.777469,...,-0.926091,-0.054602,-0.551453,-0.111063,0.797047,-0.028597,-0.161100,-2.306009,exp1_dense_baseline_pure_legacy,BaselineTransformer


In [86]:
df_merged = merge_embeddings_with_features_outcomes(
    df_features=df_features,
    df_outcome=df_outcome,
    df_embeddings=df_embeddings,
    features_key='asdb_member_key',    # Medicaid native key
    embeddings_key='individual_id',     # TE pipeline key
    verbose=True
)

NameError: name 'df_embeddings' is not defined

In [75]:
prepared_data = prepare_medicaid_evaluation_data(
    df=df_merged,
    feature_set='hybrid',
    apply_downsampling=True,
    downsample_ratio=CATBOOST_UNDERSAMPLE_RATIO,  # 0.2
    split_random_state=RANDOM_STATE,  # 35
    undersample_random_state=UNDERSAMPLE_RANDOM_STATE,  # 53
    verbose=True
)
# Create CatBoost model with tuned hyperparameters
model = CatBoostClassifier(**CATBOOST_TUNED_PARAMS)

# Train and evaluate
split_results = evaluate_model_on_splits(model, prepared_data, verbose=True)

# Extract test metrics
test_metrics = split_results['test']


PREPARING MEDICAID IP DATA FOR EVALUATION
Feature set: hybrid
Downsampling: True (ratio: 0.2)

[Step 1/4] Preprocessing features...
Preprocessing features...
  Filled 556 numeric columns with 0
  Filled 6 string columns with ''
  Encoded gender: M->1, F->0, other->-1
✅ Preprocessing complete!

[Step 2/4] Creating train/val/test splits...

Creating stratified train/val/test splits...
  train: 1,755,448 rows, 35,990 positives (2.05%)
  val: 219,432 rows, 4,499 positives (2.05%)
  test: 219,432 rows, 4,499 positives (2.05%)

[Step 3/4] Selecting features for 'hybrid'...
  Available tabular features: 243
  Available embedding features: 0
  Selected features: 243

[Step 4/4] Applying downsampling to training set...
  Downsampling: 1719458:35990 -> 143960:35990 (ratio: 0.200)
  Categorical features for CatBoost: 11

✅ Data preparation complete! (65.3s)
   Train: 179,950 samples (35,990 positives)
   Val: 219,432 samples (4,499 positives)
   Test: 219,432 samples (4,499 positives)


Training

In [79]:
from tqdm.notebook import tqdm

In [93]:
# =============================================================================
# STEP 2: Evaluate each experiment
# =============================================================================
print("\n[Step 2] Evaluating each experiment...")

# Store results for comparison
all_results = {}
for exp_name, embedding_table in tqdm(MEDICAID_EMBEDDING_TABLES.items()):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp_name}")
    print(f"{'='*70}")
    
    # Load embeddings for this experiment
    df_embeddings = load_embeddings_from_bigquery(embedding_table, verbose=True)
    
    # Merge with features and outcomes
    df_merged = merge_embeddings_with_features_outcomes(
        df_features=df_features,
        df_outcome=df_outcome,
        df_embeddings=df_embeddings,
        features_key='asdb_member_key',    # Medicaid native key
        embeddings_key='individual_id',     # TE pipeline key
        verbose=True
    )
    
    # Evaluate three feature sets: embedding_only, tabular_only, hybrid
    exp_results = {}
    
    for feature_set in ['embedding_only', 'tabular_only', 'hybrid']:
        print(f"\n--- Feature set: {feature_set} ---")
        
        # Prepare data (preprocessing, split, downsampling)
        prepared_data = prepare_medicaid_evaluation_data(
            df=df_merged,
            feature_set=feature_set,
            apply_downsampling=True,
            downsample_ratio=CATBOOST_UNDERSAMPLE_RATIO,  # 0.2
            split_random_state=RANDOM_STATE,  # 35
            undersample_random_state=UNDERSAMPLE_RANDOM_STATE,  # 53
            verbose=True
        )
        
        # Create CatBoost model with tuned hyperparameters
        model = CatBoostClassifier(**CATBOOST_TUNED_PARAMS)
        
        # Train and evaluate
        split_results = evaluate_model_on_splits(model, prepared_data, verbose=True)
        
        # Extract test metrics
        test_metrics = split_results['test']
        
        exp_results[feature_set] = {
            'auc_roc': test_metrics['auc_roc'],
            'lift_1pct': test_metrics['lift_1pct'],
            'lift_5pct': test_metrics['lift_5pct'],
            'lift_10pct': test_metrics['lift_10pct'],
            'ppv_1pct': test_metrics['ppv_1pct'],
            'sensitivity_1pct': test_metrics['sensitivity_1pct'],
            'n_features': len(prepared_data.feature_cols),
        }
        
        print(f"\n📊 {exp_name} - {feature_set}:")
        print(f"   AUC-ROC: {test_metrics['auc_roc']:.4f}")
        print(f"   Lift@1%: {test_metrics['lift_1pct']:.2f}x")
        print(f"   Lift@10%: {test_metrics['lift_10pct']:.2f}x")
        print(f"   PPV@1%: {test_metrics['ppv_1pct']:.2f}%")
    del df_embeddings
    all_results[exp_name] = exp_results


[Step 2] Evaluating each experiment...


  0%|          | 0/4 [00:00<?, ?it/s]


EXPERIMENT: exp1_dense_baseline_pure_legacy
Loading embeddings from: edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_exp1_dense_baseline_pure_legacy_medicaid_heldout_embedding
  Loaded: 2,194,312 rows, 260 columns
  Embedding dimensions: 256

Merging data...
  Features: 3,349,194 rows (key: asdb_member_key)
  Outcomes: 3,349,194 rows (key: asdb_member_key)
  Embeddings: 2,194,312 rows (key: individual_id)
  After features+outcomes merge: 3,349,194 rows
  After embeddings merge: 2,194,312 rows
  Total columns: 567
  Positive rate: 2.05%

--- Feature set: embedding_only ---

PREPARING MEDICAID IP DATA FOR EVALUATION
Feature set: embedding_only
Downsampling: True (ratio: 0.2)

[Step 1/4] Preprocessing features...
Preprocessing features...
  Filled 256 embedding columns with 0
  Filled 300 numeric columns with 0
  Filled 6 string columns with ''
  Encoded gender: M->1, F->0, other->-1
✅ Preprocessing complete!

[Step 2/4] Creating train/val/test splits...

Creating stratified train/va

In [96]:
# =============================================================================
# STEP 3: Create comparison summary
# =============================================================================
print("\n" + "="*70)
print("MEDICAID IP DOWNSTREAM EVALUATION - SUMMARY")
print("="*70)

# Build comparison DataFrame
comparison_rows = []
for exp_name, exp_results in all_results.items():
    for feature_set, metrics in exp_results.items():
        comparison_rows.append({
            'experiment': exp_name,
            'feature_set': feature_set,
            'auc_roc': metrics['auc_roc'],
            'lift_1pct': metrics['lift_1pct'],
            'lift_5pct': metrics['lift_5pct'],
            'lift_10pct': metrics['lift_10pct'],
            'ppv_1pct': metrics['ppv_1pct'],
            'sensitivity_1pct': metrics['sensitivity_1pct'],
            'n_features': metrics['n_features'],
        })

df_comparison = pd.DataFrame(comparison_rows)

# Display sorted by AUC
print("\n📊 All Results (sorted by AUC-ROC):")
print(df_comparison.sort_values('auc_roc', ascending=False).to_string(index=False))

# Show best per feature set
print("\n📊 Best per Feature Set:")
for fs in ['embedding_only', 'tabular_only', 'hybrid']:
    fs_data = df_comparison[df_comparison['feature_set'] == fs].sort_values('auc_roc', ascending=False).iloc[0]
    print(f"  {fs}: {fs_data['experiment']} (AUC={fs_data['auc_roc']:.4f}, Lift@1%={fs_data['lift_1pct']:.2f}x)")


MEDICAID IP DOWNSTREAM EVALUATION - SUMMARY

📊 All Results (sorted by AUC-ROC):
                     experiment    feature_set  auc_roc  lift_1pct  lift_5pct  lift_10pct  ppv_1pct  sensitivity_1pct  n_features
    exp2b_flash_learned_pool_v2         hybrid 0.886547  19.318221   9.780486    6.554850 39.608022         19.315403         499
 exp1_dense_baseline_opt_config         hybrid 0.885369  18.918073   9.789377    6.577077 38.787603         18.915315         499
         exp6_auxiliary_free_v3         hybrid 0.885259  19.051456   9.847171    6.588191 39.061076         19.048677         499
exp1_dense_baseline_pure_legacy         hybrid 0.880775  18.651309   9.580431    6.457049 38.240656         18.648589         499
         exp6_auxiliary_free_v3   tabular_only 0.879996  18.984765   9.660453    6.477054 38.924339         18.981996         243
    exp2b_flash_learned_pool_v2   tabular_only 0.879996  18.984765   9.660453    6.477054 38.924339         18.981996         243
exp1_dens

In [98]:
df_comparison.to_excel("exp_round5_3lob_1-5M_1epoch_128batch_dim256_medicaid_ip_downstream_eval.xlsx")

#### Feature importance

In [ ]:
# =============================================================================
# MEDICAID: SHAP Feature Importance Analysis
# =============================================================================
# Uses the hybrid feature set to quantify embedding vs tabular importance.
# Reuses df_merged from the Medicaid evaluation loop above.

prepared_medicaid_hybrid = prepare_medicaid_evaluation_data(
    df=df_merged,
    feature_set='hybrid',
    apply_downsampling=True,
    downsample_ratio=CATBOOST_UNDERSAMPLE_RATIO,
    split_random_state=RANDOM_STATE,
    undersample_random_state=UNDERSAMPLE_RANDOM_STATE,
    verbose=True,
)

# Train CatBoost for SHAP
catboost_medicaid_shap = CatBoostClassifier(**CATBOOST_TUNED_PARAMS)
train_pool_md_shap = Pool(
    prepared_medicaid_hybrid.X_train,
    prepared_medicaid_hybrid.y_train,
    cat_features=prepared_medicaid_hybrid.cat_feature_indices if prepared_medicaid_hybrid.cat_feature_indices else None,
)
val_pool_md_shap = Pool(
    prepared_medicaid_hybrid.X_val,
    prepared_medicaid_hybrid.y_val,
    cat_features=prepared_medicaid_hybrid.cat_feature_indices if prepared_medicaid_hybrid.cat_feature_indices else None,
)
catboost_medicaid_shap.fit(train_pool_md_shap, eval_set=val_pool_md_shap, verbose=0)
print("Medicaid CatBoost (hybrid) trained for SHAP analysis")

In [ ]:
# Run SHAP
medicaid_shap_df, medicaid_proportion_df = compute_shap_feature_importance(
    fitted_model=catboost_medicaid_shap,
    X_eval=prepared_medicaid_hybrid.X_test,
    feature_cols=prepared_medicaid_hybrid.feature_cols,
    embedding_features=prepared_medicaid_hybrid.embedding_features,
    top_k_list=[10, 20, 50],
    max_samples=2000,
    model_name="medicaid_catboost_hybrid",
    verbose=True,
)

# Save results
# medicaid_shap_df.to_excel("experiment_logs/medicaid_shap_feature_importance.xlsx", index=False)
# medicaid_proportion_df.to_excel("experiment_logs/medicaid_shap_embedding_proportions.xlsx", index=False)
# print("\nMedicaid SHAP results saved to experiment_logs/")